# 🔬 Django ORM & QuerySet Laboratory

> **Мета:** Зрозуміти як Django ORM перетворює Python-код на SQL,  
> як відбувається lazy evaluation, JOIN, агрегація та оптимізація запитів.

---

## Архітектурна схема

```
┌─────────────────────────────────────────────────────────────┐
│                     Django ORM Stack                        │
│                                                             │
│  Python Code          QuerySet AST         SQL String       │
│  ──────────           ────────────         ──────────       │
│  Note.objects    →    lazy tree       →    SELECT *         │
│  .filter(...)         in memory            FROM note        │
│  .exclude(...)        (no SQL yet!)        WHERE ...        │
│  .order_by(...)                            ↓                │
│                                            Database         │
│                                            ↓                │
│                                            Python Instances  │
└─────────────────────────────────────────────────────────────┘
```

---

### Моделі які використовуємо:
| Модель | Таблиця | Зв'язки |
|--------|---------|----------|
| `Note` | `hello_app_note` | FK -> User, FK -> Notebook, M2M -> Tag |
| `Notebook` | `hello_app_notebook` | FK -> User |
| `Tag` | `hello_app_tag` | FK -> User |
| `Reminder` | `hello_app_reminder` | FK -> Note |
| `TodoList` | `hello_app_todolist` | FK -> User |
| `TodoItem` | `hello_app_todoitem` | FK -> TodoList |

---
> 📚 **Документація:** [DJANGO_ORM_DEEP.md](../../DJANGO_ORM_DEEP.md)
## 🔧 0. Setup — Django Initialization

Перш ніж використовувати моделі поза `manage.py`, потрібно ініціалізувати Django.  
Це запускає:
- завантаження `settings.py`
- реєстрацію всіх `INSTALLED_APPS`
- підключення до бази даних
- запуск `AppConfig.ready()`

In [ ]:
import sys
import os

project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))

if project_root not in sys.path:
    sys.path.insert(0, project_root)

os.environ.setdefault('DJANGO_SETTINGS_MODULE', 'hello_project.settings')

# Дозволяє Django ORM працювати в Jupyter / PyCharm Notebook async-контексті
# Використовувати тільки для локального навчального notebook
os.environ["DJANGO_ALLOW_ASYNC_UNSAFE"] = "true"

import django
django.setup()

print('✅ Django готовий!')
print(f'   DB: {django.conf.settings.DATABASES["default"]["NAME"]}')
print(f'   Apps: {len(django.apps.registry.apps.get_app_configs())} зареєстровано')

In [ ]:
# Імпорт всіх моделей проекту
from django.contrib.auth.models import User
from hello_app.models import Note, Notebook, Tag, Reminder, TodoList, TodoItem

def sql(qs):
    return str(qs.query)

def run(qs, limit=5):
    print('--- SQL ---')
    print(sql(qs))
    print(f'--- Результат (перші {limit}) ---')

    results = list(qs[:limit])

    for item in results:
        print(f'  {item}')

    print(f'--- Всього: ~{qs.count()} записів ---')
    return results

print('✅ Моделі та утиліти завантажено!')

---
> 📚 **Документація:** [DJANGO_ORM_DEEP.md](../../DJANGO_ORM_DEEP.md)
## 🌱 Seed Data — Тестові дані

Створюємо тестові дані для лабораторії.  
> ⚠️ Запускай лише якщо база порожня або хочеш додати тестові записи.

In [ ]:
from django.db import transaction
from django.contrib.auth.models import User
from hello_app.models import Note, Notebook, Tag

lab_user, created = User.objects.get_or_create(
    username="lab_student",
    defaults={
        "email": "lab@example.com",
        "first_name": "Lab",
        "last_name": "Student",
    },
)

if created:
    lab_user.set_password("lab123")
    lab_user.save()
    print(f"✅ Створено юзера: {lab_user.username}")
else:
    print(f"ℹ️  Юзер вже існує: {lab_user.username}")


with transaction.atomic():
    Note.objects.filter(user=lab_user).delete()
    Notebook.objects.filter(user=lab_user).delete()
    Tag.objects.filter(user=lab_user).delete()

    notebooks = {}

    notebook_data = [
        ("Навчання Python", "#3776ab"),
        ("Django-проєкт", "#0c4b33"),
        ("Робота", "#e74c3c"),
        ("Побут", "#f39c12"),
        ("Дослідження", "#16a085"),
    ]

    for title, color in notebook_data:
        notebooks[title], _ = Notebook.objects.get_or_create(
            user=lab_user,
            title=title,
            defaults={"color": color},
        )

    tags = {}

    tag_data = [
        ("python", "#3776ab"),
        ("django", "#0c4b33"),
        ("database", "#2980b9"),
        ("bug", "#c0392b"),
        ("urgent", "#e74c3c"),
        ("meeting", "#8e44ad"),
        ("idea", "#27ae60"),
        ("article", "#2c3e50"),
        ("home", "#f39c12"),
        ("reading", "#7f8c8d"),
    ]

    for name, color in tag_data:
        tags[name], _ = Tag.objects.get_or_create(
            user=lab_user,
            name=name,
            defaults={"color": color},
        )

    notes_data = [
        {
            "title": "Виправити помилку SynchronousOnlyOperation",
            "content": "У Jupyter Django ORM падає через async context. Треба додати DJANGO_ALLOW_ASYNC_UNSAFE тільки для локального notebook.",
            "priority": 4,
            "notebook": "Django-проєкт",
            "tags": ["django", "bug", "urgent"],
            "is_pinned": True,
            "is_archived": False,
        },
        {
            "title": "Перевірити SQL через Django Debug Toolbar",
            "content": "Подивитися, які SQL-запити створює сторінка нотаток. Особливо перевірити N+1 problem.",
            "priority": 3,
            "notebook": "Django-проєкт",
            "tags": ["django", "database"],
            "is_pinned": False,
            "is_archived": False,
        },
        {
            "title": "Додати select_related для notebook",
            "content": "У списку нотаток кожна нотатка звертається до notebook. Треба оптимізувати QuerySet через select_related.",
            "priority": 4,
            "notebook": "Django-проєкт",
            "tags": ["django", "database", "bug"],
            "is_pinned": True,
            "is_archived": False,
        },
        {
            "title": "Повторити filter, exclude і order_by",
            "content": "Потрібно потренувати базові QuerySet-операції на живих прикладах із нотатками.",
            "priority": 2,
            "notebook": "Навчання Python",
            "tags": ["python", "django"],
            "is_pinned": False,
            "is_archived": False,
        },
        {
            "title": "Розібрати Q objects",
            "content": "Навчитися робити OR-запити: знайти нотатки, де title містить Django або content містить SQL.",
            "priority": 3,
            "notebook": "Навчання Python",
            "tags": ["python", "django", "database"],
            "is_pinned": False,
            "is_archived": False,
        },
        {
            "title": "Підготувати питання до зустрічі",
            "content": "На зустрічі обговорити структуру Django-уроку, seed-дані, приклади ORM і домашнє завдання.",
            "priority": 3,
            "notebook": "Робота",
            "tags": ["meeting", "django"],
            "is_pinned": False,
            "is_archived": False,
        },
        {
            "title": "Надіслати короткий звіт",
            "content": "Підготувати короткий текст про прогрес: Django models, ORM queries, templates, Bootstrap.",
            "priority": 4,
            "notebook": "Робота",
            "tags": ["urgent", "meeting"],
            "is_pinned": True,
            "is_archived": False,
        },
        {
            "title": "Купити каву і папір",
            "content": "Після роботи зайти в магазин і купити каву, папір для нотаток і батарейки.",
            "priority": 1,
            "notebook": "Побут",
            "tags": ["home"],
            "is_pinned": False,
            "is_archived": False,
        },
        {
            "title": "Заплатити за інтернет",
            "content": "Перевірити рахунок і оплатити домашній інтернет до кінця тижня.",
            "priority": 2,
            "notebook": "Побут",
            "tags": ["home"],
            "is_pinned": False,
            "is_archived": False,
        },
        {
            "title": "Ідея для статті про flood mapping",
            "content": "Порівняти SAR, optical і ML-підходи для оперативного картування паводків в умовах обмежених даних.",
            "priority": 4,
            "notebook": "Дослідження",
            "tags": ["article", "idea", "reading"],
            "is_pinned": True,
            "is_archived": False,
        },
        {
            "title": "Прочитати статтю про Sentinel-1",
            "content": "Звернути увагу на preprocessing: orbit file, calibration, speckle filtering, terrain correction.",
            "priority": 3,
            "notebook": "Дослідження",
            "tags": ["article", "reading"],
            "is_pinned": False,
            "is_archived": False,
        },
        {
            "title": "Старий план уроку ORM",
            "content": "Попередній план був занадто абстрактний. Треба замінити його живими прикладами.",
            "priority": 1,
            "notebook": "Навчання Python",
            "tags": ["python", "django"],
            "is_pinned": False,
            "is_archived": True,
        },
    ]

    for item in notes_data:
        note = Note.objects.create(
            user=lab_user,
            title=item["title"],
            content=item["content"],
            priority=item["priority"],
            notebook=notebooks[item["notebook"]],
            is_pinned=item["is_pinned"],
            is_archived=item["is_archived"],
        )

        note.tags.set([tags[tag_name] for tag_name in item["tags"]])


print("\n📊 Стан бази:")
print(f"   Notes:     {Note.objects.filter(user=lab_user).count()}")
print(f"   Notebooks: {Notebook.objects.filter(user=lab_user).count()}")
print(f"   Tags:      {Tag.objects.filter(user=lab_user).count()}")

---
> 📚 **Документація:** [DJANGO_ORM_DEEP.md](../../DJANGO_ORM_DEEP.md) · [ORM_MERMAID.md](../../ORM_MERMAID.md)

## 0. 🧱 Що таке object в Django ORM?

Перед тим як говорити про `QuerySet`, треба зрозуміти базову ідею Django ORM.

**ORM** означає **Object-Relational Mapping**.

Тобто Django бере таблиці з бази даних і дозволяє працювати з ними як з Python-обʼєктами.

```text
Database table
      ↓
Django Model class
      ↓
Python object
      ↓
SQL query
      ↓
Database rows
````

---

## 🧠 Головна ідея

У Django ORM є кілька різних “обʼєктів”, які не можна плутати між собою.

| Поняття           | Що це                            | Приклад                         |
| ----------------- | -------------------------------- | ------------------------------- |
| `Model class`     | Python-клас, який описує таблицю | `Note`                          |
| `Model instance`  | Один конкретний запис таблиці    | `note = Note.objects.first()`   |
| `Manager`         | Обʼєкт для старту запитів        | `Note.objects`                  |
| `QuerySet`        | Опис SQL-запиту, ще не результат | `Note.objects.filter(...)`      |
| `Field object`    | Опис поля моделі                 | `Note._meta.get_field("title")` |
| `Related Manager` | Обʼєкт для роботи зі звʼязками   | `note.tags.all()`               |

---

## 1. Model class — модель як опис таблиці

Коли ми пишемо:

```python
class Note(models.Model):
    title = models.CharField(...)
    content = models.TextField(...)
    priority = models.IntegerField(...)
```

ми створюємо **Python-клас**, який описує таблицю в базі даних.

Тобто:

```python
Note
```

це ще **не нотатка**.

Це **клас**, тобто схема / шаблон / інструкція для Django.

```text
Note class
   ↓
описує таблицю hello_app_note
   ↓
поля class → колонки таблиці
```

---

## 2. Model instance — один рядок таблиці як Python-обʼєкт

Коли ми отримуємо запис із бази:

```python
note = Note.objects.first()
```

ми отримуємо **model instance**.

Це вже конкретна нотатка:

```python
note.title
note.content
note.priority
```

Тобто один рядок таблиці стає Python-обʼєктом.

```text
row in database
      ↓
Note instance in Python
      ↓
note.title
note.priority
note.created_at
```

---

## 3. Manager — `objects`

Коли ми пишемо:

```python
Note.objects
```

`objects` — це **Manager**.

Manager — це спеціальний обʼєкт, через який ми починаємо запити до бази.

```python
Note.objects.all()
Note.objects.filter(priority=4)
Note.objects.create(...)
Note.objects.get(...)
```

Тобто:

```text
Note
  ↓
Note.objects
  ↓
QuerySet
  ↓
SQL
```

---

## 4. QuerySet — опис запиту, а не список

Коли ми пишемо:

```python
qs = Note.objects.filter(priority=4)
```

Django ще **не витягує дані з бази**.

Він створює обʼєкт `QuerySet`.

`QuerySet` — це як “план SQL-запиту”.

```text
QuerySet
   ├── model: Note
   ├── where: priority >= 3
   ├── order_by: ...
   └── _result_cache: None
```

SQL виконається пізніше: коли ми почнемо ітеруватися, зробимо `list(qs)`, `len(qs)`, `bool(qs)`, `count()` тощо.

---

## 5. Field object — поле моделі як обʼєкт

У Django навіть поля моделі — це теж обʼєкти.

Наприклад:

```python
Note._meta.get_field("title")
```

повертає обʼєкт поля, де Django знає:

* назву поля;
* тип поля;
* назву колонки в базі;
* чи це звʼязок;
* чи поле може бути `NULL`;
* чи має default value.

Тобто модель — це не просто набір змінних.
Це структура метаданих, яку Django використовує для генерації SQL.

---

## 6. Related objects — звʼязки між обʼєктами

Якщо `Note` має ForeignKey до `Notebook`, тоді:

```python
note.notebook
```

повертає повʼязаний обʼєкт `Notebook`.

Якщо `Note` має ManyToMany до `Tag`, тоді:

```python
note.tags.all()
```

повертає QuerySet тегів.

```text
Note object
   ├── title
   ├── priority
   ├── notebook  → Notebook object
   └── tags      → RelatedManager → QuerySet[Tag]
```

---

## 🧭 Коротка ментальна модель

```text
Model class      = опис таблиці
Model instance   = один рядок таблиці як Python object
Manager          = точка входу до запитів
QuerySet         = lazy-опис SQL-запиту
Field object     = метадані про поле
Related Manager  = доступ до повʼязаних обʼєктів
```


# Django ORM objects: базові поняття


In [ ]:

print("\n--- 1. Model class ---")
print(f"Note: {Note}")
print(f"Тип Note: {type(Note)}")
print(f"Назва таблиці в БД: {Note._meta.db_table}")
print(f"Назва app: {Note._meta.app_label}")
print(f"Назва model: {Note._meta.model_name}")


In [ ]:
print("\n--- 2. Manager: Note.objects ---")
print(f"Note.objects: {Note.objects}")
print(f"Тип Note.objects: {type(Note.objects)}")


In [ ]:
print("\n--- 3. QuerySet object ---")
qs = Note.objects.filter(user=lab_user)

print(f"qs: {qs}")
print(f"Тип qs: {type(qs)}")
print(f"Модель QuerySet: {qs.model}")
print(f"_result_cache до виконання: {qs._result_cache}")
print(f"SQL, який Django побудує:")
print(sql(qs))


In [ ]:
print("\n--- 4. Model instance: один запис як Python-об'єкт ---")

# Тут ми будуємо ORM-запит до таблиці Note.
# Note.objects — це Manager, тобто точка входу до ORM-запитів для моделі Note.
note = (
    Note.objects

    # filter(user=lab_user) додає WHERE user_id = lab_user.id.
    # Тобто ми беремо тільки нотатки конкретного користувача.
    .filter(user=lab_user)

    # select_related("notebook") робить SQL JOIN з таблицею Notebook.
    # Це потрібно, щоб note.notebook не створював додатковий SQL-запит.
    # Використовується для ForeignKey / OneToOne зв'язків.
    .select_related("notebook")

    # prefetch_related("tags") окремо завантажує ManyToMany-зв'язок tags.
    # Django зробить додатковий SQL-запит для тегів і зв'яже їх у Python.
    # Використовується для ManyToMany / reverse FK зв'язків.
    .prefetch_related("tags")

    # first() виконує SQL-запит і повертає перший об'єкт Note або None.
    # Саме тут QuerySet перестає бути lazy і реально звертається до бази.
    .first()
)

# Перевіряємо, чи ORM-запит справді повернув об'єкт Note.
# Якщо записів немає, note буде None.
if note:

    # note — це вже Model instance.
    # Тобто один конкретний рядок таблиці hello_app_note як Python-об'єкт.
    print(f"note: {note}")

    # Показуємо Python-тип об'єкта.
    # Має бути щось типу: <class 'hello_app.models.Note'>
    print(f"Тип note: {type(note)}")

    # note.pk — primary key об'єкта.
    # Це універсальний спосіб звернутися до головного ключа моделі.
    # Зазвичай pk = id.
    print(f"PK / ID: {note.pk}")

    # note.title — доступ до поля title моделі Note.
    # Це значення було прочитане з колонки title у таблиці hello_app_note.
    print(f"Title: {note.title}")

    # note.priority — доступ до поля priority моделі Note.
    # Це звичайне поле таблиці, яке Django перетворює на атрибут Python-об'єкта.
    print(f"Priority: {note.priority}")

    # note.is_pinned — boolean-поле моделі.
    # Django перетворює значення з БД на Python True / False.
    print(f"Is pinned: {note.is_pinned}")

    # note.is_archived — ще одне boolean-поле.
    # Так ми бачимо, чи нотатка архівована.
    print(f"Is archived: {note.is_archived}")


    print("\nВнутрішній словник об'єкта note.__dict__:")

    # note.__dict__ показує внутрішні атрибути Python-об'єкта.
    # Тут можна побачити, які поля реально завантажені в пам'ять.
    # Наприклад: id, user_id, notebook_id, title, content, priority.
    for key, value in note.__dict__.items():
        print(f"  {key}: {value}")


    print("\nСтан Django-об'єкта note._state:")

    # note._state — внутрішній службовий об'єкт Django.
    # Він зберігає інформацію про стан model instance.
    # Зазвичай у прикладному коді напряму його не використовують.
    print(f"  adding: {note._state.adding}")

    # note._state.adding показує, чи об'єкт ще новий і не збережений у БД.
    # False означає: об'єкт уже існує в базі.
    # True було б для нового Note(...), який ще не save().

    # note._state.db показує, з якої бази даних було завантажено об'єкт.
    # Зазвичай це "default".
    print(f"  db: {note._state.db}")

else:
    # Якщо .first() нічого не знайшов, ORM повернув None.
    # Це безпечніше, ніж .get(), бо .get() кинув би DoesNotExist.
    print("❌ Немає нотаток для lab_user. Спочатку запусти seed cell.")

In [ ]:
print("\n--- 5. Field objects: поля моделі як об'єкти ---")

# Note._meta — це внутрішній Django-об'єкт з метаданими моделі.
# Через _meta Django знає:
# - як називається таблиця;
# - які поля має модель;
# - які поля є ForeignKey / ManyToMany;
# - які колонки є в базі;
# - які reverse-зв'язки існують.
#
# get_fields() повертає список усіх полів і зв'язків моделі Note.
# Це не дані з таблиці, а ОПИС структури моделі.
for field in Note._meta.get_fields():

    # field.name — назва поля або зв'язку в Django-моделі.
    # Наприклад: id, user, notebook, title, content, tags.
    field_name = field.name

    # field.__class__.__name__ показує тип Django-поля.
    # Наприклад:
    # AutoField / BigAutoField       -> id
    # ForeignKey                     -> user, notebook
    # CharField                      -> title
    # TextField                      -> content
    # IntegerField                   -> priority
    # BooleanField                   -> is_pinned, is_archived
    # DateTimeField                  -> created_at, updated_at
    # ManyToManyField / ManyToManyRel -> tags або reverse-зв'язки
    field_type = field.__class__.__name__

    # field.is_relation показує, чи це поле є зв'язком з іншою моделлю.
    #
    # False:
    #   title, content, priority, is_pinned
    #
    # True:
    #   user, notebook, tags
    #
    # Тобто is_relation=True означає, що Django може будувати JOIN
    # або працювати через проміжну таблицю ManyToMany.
    is_relation = field.is_relation

    # Не всі field-об'єкти мають фізичну колонку в таблиці.
    #
    # Наприклад:
    # title має column="title"
    # priority має column="priority"
    # user ForeignKey має column="user_id"
    # notebook ForeignKey має column="notebook_id"
    #
    # Але ManyToMany або reverse relation можуть НЕ мати column,
    # бо вони зберігаються через окрему проміжну таблицю або є зворотним зв'язком.
    #
    # getattr(..., None) означає:
    # "візьми field.column, якщо він існує; якщо не існує — поверни None".
    column = getattr(field, "column", None)

    # Тут ми красиво друкуємо метадані поля:
    # - назву поля;
    # - тип Django field object;
    # - назву SQL-колонки;
    # - чи є це зв'язком.
    print(
        f"{field_name:15} | "
        f"type={field_type:25} | "
        f"column={str(column):20} | "
        f"relation={is_relation}"
    )


In [ ]:
print("\n--- 6. Конкретне поле title ---")
title_field = Note._meta.get_field("title")

print(f"Поле: {title_field}")
print(f"Тип поля Django: {title_field.__class__.__name__}")
print(f"Колонка в БД: {title_field.column}")
print(f"Max length: {getattr(title_field, 'max_length', None)}")
print(f"Null allowed: {title_field.null}")
print(f"Blank allowed: {title_field.blank}")


In [ ]:
print("\n--- 7. ForeignKey object: note.notebook ---")

if note and note.notebook:
    print(f"note.notebook: {note.notebook}")
    print(f"Тип note.notebook: {type(note.notebook)}")
    print(f"Notebook title: {note.notebook.title}")
else:
    print("У цієї нотатки немає notebook або нотатку не знайдено.")

In [ ]:
print("\n--- 8. ManyToMany RelatedManager: note.tags ---")

if note:
    print(f"note.tags: {note.tags}")
    print(f"Тип note.tags: {type(note.tags)}")

    tag_qs = note.tags.all()

    print(f"note.tags.all(): {tag_qs}")
    print(f"Тип tag_qs: {type(tag_qs)}")
    print(f"SQL для тегів:")
    print(sql(tag_qs))
    print(f"Теги: {[tag.name for tag in tag_qs]}")

In [ ]:
print("\n--- 9. Reverse relations для User ---")

for rel in User._meta.related_objects:
    print(
        f"Related model: {rel.related_model.__name__:15} | "
        f"field: {rel.field.name:10} | "
        f"accessor: {rel.get_accessor_name()}"
    )

---

### 🧩 Підсумок: які ORM-обʼєкти ми побачили

| Python / Django вираз | Що це означає | Роль в ORM |
|---|---|---|
| `Note` | **Model class** | Клас моделі, який описує таблицю в базі даних |
| `Note.objects` | **Manager** | Точка входу для створення ORM-запитів |
| `Note.objects.filter(...)` | **QuerySet** | Lazy-обʼєкт, який описує SQL-запит |
| `note` | **Model instance** | Один конкретний рядок таблиці як Python-обʼєкт |
| `note.notebook` | **Related model instance** | Повʼязаний обʼєкт через `ForeignKey` |
| `note.tags` | **RelatedManager** | Менеджер звʼязку через `ManyToManyField` |
| `Note._meta.fields` | **Field metadata** | Метадані про поля моделі: типи, назви колонок, звʼязки |

---

### 🧠 Ментальна модель

```text
Note
  ↓
Model class
  ↓
Note.objects
  ↓
Manager
  ↓
Note.objects.filter(...)
  ↓
QuerySet
  ↓
SQL
  ↓
Database rows
  ↓
note
  ↓
Model instance
````

---

### 🔑 Головна ідея

Django ORM складається не з одного “магічного обʼєкта”, а з кількох рівнів:

```text
Model class → Manager → QuerySet → SQL → Model instance
```

Тобто коли ти пишеш:

```python
Note.objects.filter(priority=4)
```

насправді відбувається така логіка:

```text
Note                 → модель / опис таблиці
Note.objects         → менеджер запитів
.filter(priority=4)  → побудова QuerySet
QuerySet             → lazy-опис SQL
SQL                  → виконується тільки при потребі
Result rows          → перетворюються на Note instances
```

---

### ✅ Коротко

> **Model** описує таблицю.
> **Manager** запускає запити.
> **QuerySet** описує SQL.
> **Model instance** представляє один рядок таблиці.
> **Relations** дозволяють переходити між повʼязаними таблицями як між Python-обʼєктами.

```


In [ ]:
# ──────────────────────────────────────────────────
# ДЕМОНСТРАЦІЯ LAZY EVALUATION
# Покроково показуємо: коли SQL виконується, а коли ні
# ──────────────────────────────────────────────────

print("=== Крок 1: Будуємо QuerySet (SQL ще НЕ виконується) ===")

# Note.objects.filter(user=lab_user) — будуємо AST-дерево запиту.
# Django НЕ звертається до БД прямо зараз.
# Тип qs — це <class 'django.db.models.query.QuerySet'>
# Внутрішня властивість _result_cache = None означає: дані ще не завантажені.
qs = Note.objects.filter(user=lab_user)

print(f"Тип об'єкта: {type(qs)}")
print(f"_result_cache: {qs._result_cache}")  # None = не виконано!

print("\n=== Крок 2: Додаємо фільтр (все ще lazy) ===")

# filter(priority__gte=3) — ДОДАЄМО ще одну умову до QuerySet.
# Але qs2 — це все ще lazy. SQL все ще не виконується.
# __gte означає: >= 3 (greater than or equal)
# Django просто будує WHERE priority >= 3 AND user_id = ...
qs2 = qs.filter(priority__gte=3)

# sql(qs2) — наш helper-метод, що показує SQL без виконання запиту.
# Він викликає str(qs2.query) — просто рядок SQL, не реальний запит до БД.
print(f"SQL буде: {sql(qs2)}")
print(f"_result_cache: {qs2._result_cache}")  # Все ще None!

print("\n=== Крок 3: list() ВИКОНУЄ SQL ===")

# list(qs2) — ПРИМУШУЄ виконати SQL-запит.
# Django: 1) будує SQL, 2) відправляє до SQLite/PostgreSQL,
#         3) отримує rows, 4) конвертує кожен row у Model instance.
# Результат зберігається у qs2._result_cache = [Note, Note, ...]
results = list(qs2)  # <-- ТУТ іде SQL запит!

print(f"_result_cache: {qs2._result_cache}")  # Тепер заповнений!
print(f"Знайдено: {len(results)} нотаток з priority >= 3")

print("\n=== Крок 4: Повторна ітерація = з кешу (0 SQL) ===")

# Після того як _result_cache заповнений, повторний list() або for-цикл
# НЕ йде до бази даних. Дані беруться з Python-пам'яті (_result_cache).
# Це і є QuerySet caching — але тільки для вже виконаних QuerySet!
results2 = list(qs2)  # Використовує _result_cache, НЕ йде до БД!

print("Результат з кешу, SQL не виконувався")


In [ ]:
# ──────────────────────────────────────────────────
# all(), count(), exists()
# ──────────────────────────────────────────────────

print("=== all(), count(), exists() ===")


# Note.objects — Manager моделі Note.
# Через нього ми починаємо ORM-запит до таблиці hello_app_note.
#
# filter(user=lab_user) додає SQL-умову:
# WHERE user_id = lab_user.id
#
# all() повертає QuerySet.
# Важливо: all() НЕ виконує SQL одразу.
# Це все ще lazy QuerySet — тільки опис майбутнього запиту.
all_notes = Note.objects.filter(user=lab_user).all()

# sql(all_notes) показує SQL, який Django побудував.
# Але самі записи ще не завантажені з бази.
print("all():")
print(all_notes)
print(sql(all_notes))


# count() — це вже НЕ lazy.
# count() одразу виконує SQL-запит типу:
# SELECT COUNT(*) FROM hello_app_note WHERE user_id = ...
#
# Django не завантажує всі Note-об'єкти в Python.
# База просто рахує кількість рядків і повертає число.
count = Note.objects.filter(user=lab_user).count()

print(
    f"\ncount() = {count} "
    "(1 SQL: SELECT COUNT(*), без завантаження об'єктів)"
)


# exists() — це теж одразу виконує SQL.
# Але він не рахує всі записи.
#
# exists() перевіряє тільки факт наявності хоча б одного рядка.
# SQL приблизно такий:
# SELECT 1 FROM hello_app_note WHERE ... LIMIT 1
#
# Це швидше, ніж count(), якщо тобі треба тільки True / False.
has_urgent = Note.objects.filter(
    user=lab_user,
    priority=4,
).exists()

print(
    f"exists(priority=4) = {has_urgent} "
    "(SELECT 1 ... LIMIT 1, без завантаження об'єктів)"
)

In [ ]:
# ──────────────────────────────────────────────────
# Типи об'єктів, які повертають ORM-методи
# ──────────────────────────────────────────────────

print("\n=== Типи об'єктів ===")

# .all() — повертає QuerySet (lazy).
# Тип: <class 'django.db.models.query.QuerySet'>
# _result_cache = None — SQL ще не виконаний.
all_notes = Note.objects.filter(user=lab_user).all()

# .count() — одразу виконує SQL.
# SQL: SELECT COUNT(*) FROM hello_app_note WHERE user_id = ...
# Повертає: int (Python-число, НЕ QuerySet)
count = Note.objects.filter(user=lab_user).count()

# .exists() — одразу виконує SQL.
# SQL: SELECT 1 FROM hello_app_note WHERE user_id=... AND priority=4 LIMIT 1
# Повертає: bool (True / False, НЕ QuerySet)
# Набагато швидше ніж .count() > 0, бо зупиняється на першому рядку.
has_urgent = Note.objects.filter(user=lab_user, priority=4).exists()

print(f"all_notes: {all_notes}")

# all_notes[0] — ІНДЕКСУВАННЯ. Це виконує SQL!
# SQL: SELECT ... LIMIT 1 OFFSET 0
# Повертає: один Model instance (один рядок таблиці як Python-об'єкт).
# ВАЖЛИВО: після [0] кеш НЕ заповнюється для решти об'єктів!
print(f"first_notes: {all_notes[0]}")

print(f"Тип all_notes: {type(all_notes)}")         # QuerySet
print(f"_result_cache: {all_notes._result_cache}")  # None або список

print(f"\ncount: {count}")
print(f"Тип count: {type(count)}")                  # <class 'int'>

print(f"\nhas_urgent: {has_urgent}")
print(f"Тип has_urgent: {type(has_urgent)}")        # <class 'bool'>


In [ ]:
# ──────────────────────────────────────────────────
# filter() та exclude()
# ──────────────────────────────────────────────────
print("=== filter() ===")
# Note.objects — Manager моделі Note.
# Через нього ми починаємо ORM-запит до таблиці hello_app_note.
# filter(...) додає SQL-умови через AND.
# user=lab_user означає:
# WHERE user_id = lab_user.id
# priority__gte=3 означає:
# priority >= 3
# __gte — це Django lookup:
# gte = greater than or equal = більше або дорівнює.
# Важливо:
# цей рядок ще НЕ виконує SQL.
# Він тільки створює QuerySet.
qs_high = Note.objects.filter(
    user=lab_user,
    priority__gte=3,
)
# Тут ми просто друкуємо текстовий опис прикладу.
print("filter(priority__gte=3):")
# sql(qs_high) показує SQL, який Django побудував з QuerySet.
# Але сам QuerySet ще не обовʼязково був виконаний.
print(f"SQL: {sql(qs_high)}")
# Цикл for запускає виконання QuerySet.
# Саме тут Django реально звертається до бази даних.
# SQL виконується один раз, результати перетворюються на Note instances.
for n in qs_high:
    # n — це вже Model instance, тобто один рядок таблиці як Python-обʼєкт.
    # n.priority читає значення колонки priority.
    # n.title читає значення колонки title.
    print(f"  [{n.priority}] {n.title}")

print("\n=== exclude() ===")
# Note.objects.filter(user=lab_user) створює базовий QuerySet:
# всі нотатки конкретного користувача.
# .exclude(priority=1) додає SQL-заперечення:
# WHERE NOT priority = 1
# Тобто ми беремо всі нотатки користувача,
# крім тих, у яких priority дорівнює 1.
#
# Важливо:
# exclude() теж повертає QuerySet і теж є lazy.
qs_not_low = (
    Note.objects
    .filter(user=lab_user)
    .exclude(priority=1)
)

print("exclude(priority=1):")

# Показуємо SQL-запит, який Django згенерував.
# Тут має бути умова NOT або NOT (... priority = 1 ...).
print(f"SQL: {sql(qs_not_low)}")


# Ітерація запускає SQL-запит.
# Django отримує рядки з бази і створює Python-обʼєкти Note.
for n in qs_not_low:
    print(f"  [{n.priority}] {n.title}")

In [ ]:
# ──────────────────────────────────────────────────
# get() та filter().first()
# ──────────────────────────────────────────────────
print("=== get() — одразу виконує SQL ===")
try:
    # Note.objects — Manager моделі Note.
    # Через нього ми починаємо ORM-запит до таблиці hello_app_note.
    # get(...) шукає РІВНО ОДИН запис у базі даних.
    # user=lab_user означає:
    # WHERE user_id = lab_user.id
    # title="..." означає:
    # AND title = "Виправити помилку SynchronousOnlyOperation"
    # ВАЖЛИВО:
    # get() НЕ є lazy.
    # Він одразу виконує SQL-запит.
    # Якщо знайдено 1 запис  -> повертає Note instance.
    # Якщо знайдено 0 записів -> кидає Note.DoesNotExist.
    # Якщо знайдено 2+ записи -> кидає Note.MultipleObjectsReturned.
    note = Note.objects.get(
        user=lab_user,
        title="Виправити помилку SynchronousOnlyOperation",
    )
    # Якщо ми дійшли сюди, значить get() знайшов рівно один запис.
    # note — це вже Model instance:
    # один рядок таблиці hello_app_note як Python-об'єкт.
    print(f"Знайдено: {note.title} (priority={note.priority})")
# Цей except спрацює, якщо get() не знайшов жодного запису.
# Наприклад, якщо title неправильний або seed-дані не були створені.
except Note.DoesNotExist:
    print("❌ DoesNotExist: запис не знайдено")
# Цей except спрацює, якщо get() знайшов більше одного запису.
# get() завжди очікує тільки один об'єкт.
# Якщо умова недостатньо точна, Django не знає, який саме запис повернути.
except Note.MultipleObjectsReturned:
    print("❌ MultipleObjectsReturned: знайдено більше одного запису!")

print("\n=== filter().first() — безпечніший варіант ===")
# filter(...) повертає QuerySet.
# На відміну від get(), filter() може знайти:
# - 0 записів;
# - 1 запис;
# - багато записів.
## filter() сам по собі lazy — SQL ще не виконується.
note_qs = Note.objects.filter(
    user=lab_user,
    title="Виправити помилку SynchronousOnlyOperation",
)
# first() бере перший запис із QuerySet.
# Саме first() виконує SQL-запит.
# SQL буде приблизно:
# SELECT ... WHERE ... ORDER BY ... LIMIT 1
# Якщо запис є     -> повертає Note instance.
# Якщо запису нема -> повертає None.
# Тобто first() не кидає DoesNotExist.
note_safe = note_qs.first()
# Тут ми друкуємо результат.
# Якщо запис знайдено, буде надруковано об'єкт Note.
# Якщо не знайдено, буде None.
print(
    f"\nfilter().first() = {note_safe} "
    "(None якщо не знайдено, не Exception)"
)

---
> 📚 **Документація:** [DJANGO_ORM_DEEP.md](../../DJANGO_ORM_DEEP.md) · [RELATIONAL_DB_FOUNDATIONS.md](../../RELATIONAL_DB_FOUNDATIONS.md)
## 3. 🔎 Field Lookups — Оператори фільтрації

```
field__lookup=value
   │       └── тип порівняння
   └── назва поля
```

| Lookup | SQL | Приклад |
|--------|-----|---------|
| `exact` | `= value` | `priority__exact=4` |
| `iexact` | `LIKE 'value'` (case-insensitive) | `title__iexact='django'` |
| `contains` | `LIKE '%value%'` | `content__contains='SQL'` |
| `icontains` | `LIKE '%value%'` (CI) | `title__icontains='orm'` |
| `startswith` | `LIKE 'value%'` | `title__startswith='Query'` |
| `endswith` | `LIKE '%value'` | `title__endswith='SQL'` |
| `gt` / `gte` | `>` / `>=` | `priority__gte=3` |
| `lt` / `lte` | `<` / `<=` | `priority__lt=3` |
| `in` | `IN (v1, v2)` | `priority__in=[3,4]` |
| `isnull` | `IS NULL` | `notebook__isnull=True` |
| `range` | `BETWEEN a AND b` | `id__range=(1, 10)` |

In [ ]:
# ──────────────────────────────────────────────────
# Field Lookups — оператори фільтрації
# Синтаксис: field__lookup=value
# ──────────────────────────────────────────────────

# __icontains — пошук рядка без урахування регістру
# SQL: WHERE title ILIKE '%sql%' (PostgreSQL) або LIKE '%sql%' (SQLite)
# "i" = case-Insensitive (регістронезалежний)
print('=== icontains — регістронезалежний пошук ===')
qs = Note.objects.filter(user=lab_user, title__icontains='sql')
print(f'SQL: {sql(qs)}')
print(f'Знайдено: {[n.title for n in qs]}')

# __in — перевіряє чи поле є у списку значень
# SQL: WHERE priority IN (3, 4)
# Корисно замість кількох OR-умов: filter(priority=3) | filter(priority=4)
print('\n=== in — перевірка приналежності до списку ===')
qs = Note.objects.filter(user=lab_user, priority__in=[3, 4])
print(f'SQL: {sql(qs)}')
print(f'Знайдено: {[f"[{n.priority}]{n.title}" for n in qs]}')

# __isnull — перевіряє NULL / NOT NULL
# SQL: WHERE notebook_id IS NULL
# notebook — nullable ForeignKey (null=True, blank=True)
# Нотатки де notebook_id = NULL — не прив'язані до жодного записника
print('\n=== isnull — без записника ===')
qs = Note.objects.filter(user=lab_user, notebook__isnull=True)
print(f'SQL: {sql(qs)}')
print(f'Без записника: {qs.count()}')

# Ланцюжок фільтрів (chaining) — кожен .filter() додає AND-умову
# Django будує одне SQL WHERE з усіма умовами разом:
# WHERE user_id=... AND priority >= 2 AND is_archived=False AND title ILIKE '%django%'
# .exclude(is_archived=True) генерує: AND NOT is_archived = True
print('\n=== Ланцюжок фільтрів (chaining) ===')
qs = (Note.objects
      .filter(user=lab_user)
      .filter(priority__gte=2)
      .exclude(is_archived=True)
      .filter(title__icontains='django'))
print(f'SQL: {sql(qs)}')
print(f'Результат: {[n.title for n in qs]}')


---
> 📚 **Документація:** [DJANGO_ORM_DEEP.md](../../DJANGO_ORM_DEEP.md) · [ORM_MERMAID.md](../../ORM_MERMAID.md)
## 4. 🔗 Relationship Traversal — __ для JOIN

Django використовує `__` для обходу зв'язків.  
Кожен `__` між назвами полів = SQL `JOIN`.

```
Note.objects.filter(notebook__title='Django')
                        │
                    JOIN notebook ON note.notebook_id = notebook.id
                             │
                         WHERE notebook.title = 'Django'
```

```
Модель   Поле     Модель    Поле
  Note --notebook--> Notebook --title-->'Django'
           FK            JOIN
```

In [ ]:
# ──────────────────────────────────────────────────
# Relationship Traversal — traversal через __
# Django автоматично будує SQL JOIN при traversal
# ──────────────────────────────────────────────────

# Forward FK: Note -> Notebook
# notebook__title — Django бачить "notebook" як ForeignKey поле Note.
# Тому: JOIN hello_app_notebook ON hello_app_note.notebook_id = hello_app_notebook.id
# Потім: WHERE hello_app_notebook.title = 'Django-проєкт'
# Все одним SQL запитом — Django сам вирішує тип JOIN.
print('=== Forward FK: Note -> Notebook ===')
qs = Note.objects.filter(user=lab_user, notebook__title="Django-проєкт")
print(f'SQL:\n{sql(qs)}')
print(f'Нотатки у записнику Django: {[n.title for n in qs]}')

# Двоступеневий traversal: Note -> Notebook -> User
# Django будує 2 JOIN:
# JOIN hello_app_notebook ON note.notebook_id = notebook.id
# JOIN auth_user ON notebook.user_id = auth_user.id
# WHERE auth_user.username = 'lab_student'
print('\n=== Forward FK глибше: Note -> Notebook -> User ===')
qs = Note.objects.filter(notebook__user__username='lab_student')
print(f'SQL:\n{sql(qs)}')
print(f'Результат: {qs.count()} нотаток')

# M2M traversal: Note -> Tag
# ManyToManyField автоматично будує JOIN через junction таблицю:
# JOIN hello_app_note_tags ON note.id = note_tags.note_id
# JOIN hello_app_tag ON note_tags.tag_id = tag.id
# WHERE hello_app_tag.name = 'orm'
print('\n=== M2M: Note -> Tag ===')
qs = Note.objects.filter(user=lab_user, tags__name='orm')
print(f'SQL:\n{sql(qs)}')
print(f'Нотатки з тегом orm: {[n.title for n in qs.distinct()]}')

# УВАГА: M2M JOIN може давати дублікати!
# Якщо нотатка має 2 теги 'orm' і 'sql', то після JOIN вона з'являється 2 рази.
# Рішення: .distinct() — додає DISTINCT у SELECT, прибирає дублікати.
# SQL: SELECT DISTINCT ... WHERE ...
print('\n=== ПРОБЛЕМА: M2M + кілька тегів = дублікати! ===')
qs_dup = Note.objects.filter(user=lab_user, tags__name__in=['orm', 'sql'])
print(f'Без distinct(): {qs_dup.count()} (може бути дублікати!)')
qs_ok = qs_dup.distinct()
print(f'З distinct():   {qs_ok.count()} (унікальні)')


In [ ]:
# ──────────────────────────────────────────────────
# Reverse FK — доступ до дочірніх об'єктів через related_name
# ──────────────────────────────────────────────────

# Reverse FK: з Notebook -> Note
# У models.py поле Note.notebook має related_name='notes'.
# Тому: notebook.notes — це RelatedManager (не QuerySet відразу).
# notebook.notes.all() — повертає QuerySet всіх Note з notebook_id=notebook.id
# SQL: SELECT * FROM hello_app_note WHERE notebook_id = <notebook.pk>
print("=== Reverse FK: Notebook -> Note ===")

# .first() — виконує SQL: SELECT ... LIMIT 1. Повертає Model instance або None.
notebook = Notebook.objects.filter(user=lab_user, title="Django").first()

if notebook:
    # notebook.notes — це RelatedManager (не список, не QuerySet).
    # .all() перетворює його на QuerySet — але ще LAZY (SQL не виконаний).
    notes_in_nb = notebook.notes.all()

    print(f"Записник: {notebook.title}")
    # for n in notes_in_nb — цикл ВИКОНУЄ SQL.
    # Django: SELECT * FROM hello_app_note WHERE notebook_id = X
    print(f"Нотатки (через .notes): {[n.title for n in notes_in_nb]}")
    print(f"SQL: {sql(notes_in_nb)}")


# User -> Notes (зворотній FK через auth.User)
# У моделі Note є поле user = ForeignKey(User, related_name='notes').
# Тому lab_user.notes — RelatedManager для всіх нотаток цього user.
# .filter(priority__gte=3) — додає умову WHERE priority >= 3.
# SQL: SELECT * FROM hello_app_note WHERE user_id = X AND priority >= 3
print("\n=== User -> Notes (зворотній FK через auth.User) ===")

user_notes = lab_user.notes.filter(priority__gte=3)

print(f"Нотатки user із priority >= 3: {user_notes.count()}")
print(f"SQL: {sql(user_notes)}")


---
> 📚 **Документація:** [DJANGO_ORM_DEEP.md](../../DJANGO_ORM_DEEP.md)
## 5. 🎯 Q() Objects — Складні умови: OR, AND, NOT

Стандартний `filter()` підтримує тільки `AND`.  
Для `OR` і `NOT` потрібні `Q()` об'єкти.

```python
from django.db.models import Q

Q(priority=4) | Q(is_pinned=True)   # OR
Q(priority=4) & Q(is_pinned=True)   # AND (те саме що filter(priority=4, is_pinned=True))
~Q(priority=1)                       # NOT (те саме що exclude(priority=1))
```

In [ ]:
# ──────────────────────────────────────────────────
# Q() Objects — складні умови: OR, AND, NOT
# ──────────────────────────────────────────────────

from django.db.models import Q

# Стандартний filter() підтримує тільки AND:
# filter(a=1, b=2) → WHERE a=1 AND b=2
#
# Q() дозволяє будь-які логічні комбінації:
# Q(a=1) | Q(b=2) → WHERE (a=1 OR b=2)
# Q(a=1) & Q(b=2) → WHERE (a=1 AND b=2)
# ~Q(a=1)         → WHERE NOT a=1

# Q(priority=4) | Q(title__icontains='SQL') →
# SQL: WHERE (priority=4 OR title LIKE '%SQL%') AND user_id = lab_user.id
# Оператор | між двома Q() утворює OR в SQL.
print('=== Q() OR ===')
qs = Note.objects.filter(
    user=lab_user
).filter(
    Q(priority=4) | Q(title__icontains='SQL')
)
print(f'SQL: {sql(qs)}')
print(f'Результат: {[f"[{n.priority}]{n.title}" for n in qs]}')

# ~Q(priority=1) → SQL: NOT priority=1
# ~Q(priority=2) → SQL: NOT priority=2
# & між ними → AND
# SQL: WHERE NOT priority=1 AND NOT priority=2 AND user_id=...
print('\n=== Q() NOT ===')
qs = Note.objects.filter(
    user=lab_user,
    **{'priority__gte': 1}
).filter(~Q(priority=1) & ~Q(priority=2))
print(f'SQL: {sql(qs)}')
print(f'Результат (priority 3,4): {[f"[{n.priority}]{n.title}" for n in qs]}')

# Комбінований запит:
# (Q(priority=4) | Q(is_pinned=True)) — дужки важливі! Спочатку OR, потім AND з NOT.
# & ~Q(is_archived=True) → AND NOT is_archived=True
# SQL: WHERE (priority=4 OR is_pinned=TRUE) AND NOT is_archived=TRUE AND user_id=...
print('\n=== Комбінований запит ===')
qs = Note.objects.filter(
    user=lab_user
).filter(
    (Q(priority=4) | Q(is_pinned=True)) & ~Q(is_archived=True)
)
print(f'SQL: {sql(qs)}')
print(f'Результат: {qs.count()} нотаток')


---
> 📚 **Документація:** [DJANGO_ORM_DEEP.md](../../DJANGO_ORM_DEEP.md) · [TRANSACTIONS_CONCURRENCY.md](../../TRANSACTIONS_CONCURRENCY.md)
## 6. 📐 F() Expressions — Поля порівнюємо між собою

`F()` дозволяє посилатись на **значення поля в базі даних** без завантаження в Python.  
Це критично для атомарності (race condition safe!) і продуктивності.

```
# НЕПРАВИЛЬНО: race condition!
note = Note.objects.get(pk=1)     # SELECT (завантажили в Python)
note.priority = note.priority + 1  # Python операція
note.save()                        # UPDATE (інший запит може змінити між SELECT і UPDATE!)

# ПРАВИЛЬНО: атомарно в одному SQL
Note.objects.filter(pk=1).update(priority=F('priority') + 1)
# SQL: UPDATE note SET priority = priority + 1 WHERE id = 1
```

In [ ]:
# ──────────────────────────────────────────────────
# F() Expressions — посилання на поля моделі в SQL
# ──────────────────────────────────────────────────

from django.db.models import F
from django.db.models import ExpressionWrapper, IntegerField

# F("created_at") — посилання на значення поля created_at прямо в SQL.
# Замість: Python читає created_at та updated_at → порівнює в Python,
# Django робить: WHERE hello_app_note.updated_at = hello_app_note.created_at
# Тобто порівняння відбувається ALL у БД, без завантаження даних у Python.
# Корисно: атомарно, без race condition, ефективно.
print("=== F() для порівняння двох полів ===")

qs = Note.objects.filter(user=lab_user, updated_at=F("created_at"))

print(f"SQL: {sql(qs)}")
print(f"Нотатки що ніколи не оновлювались: {qs.count()}")


# F() у UPDATE — атомарна арифметика на рівні БД
# ПРОБЛЕМА без F(): читаємо priority=2 в Python, обчислюємо 2+1=3, пишемо 3.
# Якщо два запити читають одночасно → race condition: обидва запишуть 3!
# РІШЕННЯ з F(): SQL: UPDATE hello_app_note SET priority = priority + 1 WHERE ...
# БД сама збільшує значення — атомарно, без race condition.
print("\n=== F() у UPDATE (атомарна операція) ===")

update_qs = Note.objects.filter(
    user=lab_user,
    title="Заплатити за інтернет",
)

note = update_qs.first()

if note:
    print(f"Поточний priority: {note.priority}")
    print("SQL концепція: UPDATE hello_app_note SET priority = priority + 1 WHERE ...")

    # Безпечна демонстрація:
    # update_qs.update(priority=F("priority") + 1)  ← розкоментуй щоб виконати
    # після update_qs.update() треба: note.refresh_from_db() — перечитати з БД
    # бо Python-об'єкт note містить старе значення priority!
else:
    print("❌ Нотатку не знайдено")


# F() в annotate() — створює нове "обчислене" поле для кожного запису.
# ExpressionWrapper потрібен щоб вказати output_field (тип результату).
# SQL: SELECT ..., priority * 2 AS priority_doubled FROM hello_app_note WHERE ...
# Результат: кожен Note отримує атрибут priority_doubled — Python-int.
print("\n=== F() в annotate() ===")

qs = Note.objects.filter(user=lab_user).annotate(
    priority_doubled=ExpressionWrapper(
        F("priority") * 2,
        output_field=IntegerField()
    )
)
for n in qs[:3]:
    print(f"  {n.title}: priority={n.priority}, doubled={n.priority_doubled}")


---
> 📚 **Документація:** [DJANGO_ORM_DEEP.md](../../DJANGO_ORM_DEEP.md)
## 7. 📊 Aggregation & Annotation

| Метод | SQL | Повертає |
|-------|-----|----------|
| `aggregate()` | `SELECT AVG(field)` over ALL | `dict` |
| `annotate()` | `SELECT COUNT(*) GROUP BY id` per-row | `QuerySet` |

```
aggregate()  → одне значення для всього QuerySet
annotate()   → нове поле для КОЖНОГО запису (GROUP BY)
```

In [ ]:
# ──────────────────────────────────────────────────
# aggregate() — підсумкові функції по всьому QuerySet
# ──────────────────────────────────────────────────

from django.db.models import Avg, Max, Min, Sum, Count

# aggregate() — одразу ВИКОНУЄ SQL і повертає словник.
# НЕ lazy! Одразу йде запит до БД.
# SQL:
# SELECT AVG(priority) as avg_priority,
#        MAX(priority) as max_priority,
#        MIN(priority) as min_priority,
#        COUNT(id) as total
# FROM hello_app_note
# WHERE user_id = lab_user.id
#
# Повертає: dict — { 'avg_priority': 2.5, 'max_priority': 4, ... }
# НЕ QuerySet, НЕ Model instance — просто Python dict з числами.
print('=== aggregate() — одне значення для всіх нотаток ===')
stats = Note.objects.filter(user=lab_user).aggregate(
    avg_priority=Avg('priority'),   # AVG(priority) → float
    max_priority=Max('priority'),   # MAX(priority) → int
    min_priority=Min('priority'),   # MIN(priority) → int
    total=Count('id'),              # COUNT(id) → int
)
# stats — це звичайний Python dict.
# stats['avg_priority'] — float, stats['total'] — int.
print(f'Результат: {stats}')
print(f'  Середній пріоритет: {stats["avg_priority"]:.2f}')
print(f'  Максимальний:       {stats["max_priority"]}')
print(f'  Загальна кількість: {stats["total"]}')


In [ ]:
# ──────────────────────────────────────────────────
# annotate() — обчислене поле для КОЖНОГО запису
# GROUP BY під капотом
# ──────────────────────────────────────────────────

# annotate() відрізняється від aggregate():
# aggregate() → 1 рядок результату (підсумок по всій таблиці)
# annotate()  → N рядків (кожен об'єкт отримує нове обчислене поле)
#
# SQL для нижнього запиту:
# SELECT hello_app_notebook.*, COUNT(hello_app_note.id) AS note_count
# FROM hello_app_notebook
# LEFT JOIN hello_app_note ON note.notebook_id = notebook.id
# WHERE notebook.user_id = lab_user.id
# GROUP BY notebook.id          ← автоматично додається Django!
# ORDER BY note_count DESC
print('=== annotate() — кількість нотаток для кожного записника ===')
from django.db.models import Count

notebooks_with_count = (
    Notebook.objects
    .filter(user=lab_user)
    # note_count — нова назва поля. 'notes' — related_name зворотнього FK.
    # Django знає: для COUNT використати junction через 'notes' → notebook_id.
    .annotate(note_count=Count('notes'))
    .order_by('-note_count')  # DESC по note_count
)

print(f'SQL: {sql(notebooks_with_count)}')
print()
for nb in notebooks_with_count:
    # nb.note_count — це Python int, доданий annotate().
    # НЕ існує у моделі Notebook! Це "тимчасове" поле тільки для цього QuerySet.
    print(f'  📁 {nb.title}: {nb.note_count} нотаток')

# Те саме для нотаток — кількість тегів кожної нотатки.
# 'tags' — ManyToManyField. Django будує COUNT через junction hello_app_note_tags.
# SQL: GROUP BY note.id → COUNT(note_tags.tag_id) AS tag_count
print('\n=== annotate() — кількість тегів для кожної нотатки ===')
notes_with_tags = (
    Note.objects
    .filter(user=lab_user)
    .annotate(tag_count=Count('tags'))
    .order_by('-tag_count')
)
for n in notes_with_tags:
    print(f'  [{n.priority}] {n.title}: {n.tag_count} тегів')


In [ ]:
# ──────────────────────────────────────────────────
# values() + annotate() — GROUP BY як у чистому SQL
# ──────────────────────────────────────────────────

# values('priority') ПЕРЕД annotate() → Django будує GROUP BY priority.
# Без values(): GROUP BY була б по id (кожен рядок окремо).
# З values('priority'): всі рядки з однаковим priority об'єднуються.
#
# SQL:
# SELECT priority, COUNT(id) AS count, COUNT(id) AS avg_length
# FROM hello_app_note
# WHERE user_id = lab_user.id
# GROUP BY priority
# ORDER BY priority
#
# ВАЖЛИВО: values() перед annotate() = Django Group By mode.
# Повертає: QuerySet dict-ів (не Model instances!)
# Кожен рядок — словник: {'priority': 2, 'count': 3, ...}
print('=== values() + annotate() — GROUP BY priority ===')
priority_stats = (
    Note.objects
    .filter(user=lab_user)
    .values('priority')                     # GROUP BY priority
    .annotate(
        count=Count('id'),                  # COUNT(id) для кожної групи
        avg_length=Avg('title__length') if False else Count('id')  # спрощено
    )
    .order_by('priority')
)

priority_labels = {1: 'Низький', 2: 'Середній', 3: 'Високий', 4: 'Терміново'}
print(f'SQL: {sql(priority_stats)}')
print()
for row in priority_stats:
    # row — це dict: {'priority': 1, 'count': 3, ...}
    # НЕ Model instance! Немає .title, .content тощо.
    label = priority_labels.get(row['priority'], '?')
    bar = '█' * row['count']
    print(f'  {label:12} [{row["priority"]}]: {bar} ({row["count"]})')


---
> 📚 **Документація:** [DJANGO_ORM_DEEP.md](../../DJANGO_ORM_DEEP.md) · [INDEXING_DEEP.md](../../INDEXING_DEEP.md)
## 8. ⚡ N+1 Problem & Optimization

### What is the N+1 Problem?

The N+1 problem is the most common performance mistake in Django ORM. It occurs when
the ORM executes **1 initial query** to fetch a list of objects, then fires **N
additional queries** — one per object — to fetch a related field inside a loop.

```
Code:
    notes = Note.objects.filter(user=user)  # Query 1 — SELECT * FROM note

    for note in notes:
        print(note.notebook.title)          # Queries 2…N+1
                                            # SELECT * FROM notebook WHERE id=?
                                            # ↑ fires ONCE PER NOTE!

Result:
    10 notes   →  11 SQL queries
    100 notes  → 101 SQL queries
    1000 notes → 1001 SQL queries  😱
```

The root cause: Django FK fields are **lazy by default**. Accessing `note.notebook`
on an un-optimised QuerySet triggers a fresh `SELECT` for each object because the
related data was not included in the original query.

---

### The Solution Toolkit

| Method | Relationship | SQL Strategy | Queries |
|--------|-------------|--------------|:-------:|
| `select_related('notebook')` | `ForeignKey` / `OneToOne` | SQL `JOIN` — same query | **1** |
| `prefetch_related('tags')` | `ManyToMany` / Reverse FK | Separate SQL + Python join | **2** |
| `Prefetch(queryset=..., to_attr=...)` | M2M / Reverse FK | Filtered sub-QuerySet | **2** |
| `only('id', 'title')` | Any | Restrict fetched columns | **1** (fewer cols) |
| `.exists()` / `Exists()` subquery | Boolean check | `SELECT 1 LIMIT 1` | **1** |

---

### select_related() — SQL JOIN Strategy

`select_related()` tells Django to perform a SQL `JOIN` and include the related
object's columns in the **same** query. Works only for **single-valued** relationships
(`ForeignKey`, `OneToOne`).

```python
# ❌ N+1: 1 + N queries
notes = Note.objects.filter(user=user)
for note in notes:
    print(note.notebook.title)      # new SELECT per note

# ✅ select_related: always 1 query with LEFT OUTER JOIN
notes = Note.objects.filter(user=user).select_related('notebook')
for note in notes:
    print(note.notebook.title)      # already in memory — 0 extra SQL

# Multi-level traversal: chain FKs with __
notes = Note.objects.select_related('notebook', 'user__profile')
# SQL: SELECT note.*, notebook.*, auth_user.*, userprofile.*
#      FROM note
#      LEFT JOIN notebook    ON note.notebook_id = notebook.id
#      LEFT JOIN auth_user   ON note.user_id     = auth_user.id
#      LEFT JOIN userprofile ON auth_user.id      = userprofile.user_id
```

---

### prefetch_related() — Python Join Strategy

`prefetch_related()` fires **2 separate queries** then joins the results in Python
memory. Required for **multi-valued** relationships (`ManyToMany`, reverse `ForeignKey`).

> **Why not a JOIN for M2M?**
> A JOIN on a M2M multiplies rows — 100 notes × 5 tags each = 500 rows returned.
> Django avoids this "row explosion" by firing a second targeted query and joining
> in Python memory. Result: always exactly 2 queries, regardless of note or tag count.

```python
# ✅ prefetch_related: 2 queries total, regardless of how many notes
notes = Note.objects.filter(user=user).prefetch_related('tags')

# Query 1: SELECT * FROM note WHERE user_id = ?
# Query 2: SELECT tag.*, note_tags.note_id
#           FROM tag
#           INNER JOIN note_tags ON tag.id = note_tags.tag_id
#           WHERE note_tags.note_id IN (1, 2, 3, ...)
# Django joins result in Python → stores in QuerySet cache

for note in notes:
    print(note.tags.all())    # reads from cache — 0 extra SQL
```

---

### Prefetch() Object — Advanced Filtering with to_attr

When you need to **filter**, **order**, or **select only columns** from the prefetched
relation, use the explicit `Prefetch()` object:

```python
from django.db.models import Prefetch

notes = Note.objects.prefetch_related(
    Prefetch(
        'reminders',
        queryset=Reminder.objects.filter(is_sent=False).order_by('remind_at'),
        to_attr='pending_reminders'   # stored as a plain list, not a QuerySet
    )
)

for note in notes:
    # ✅ note.pending_reminders — already-filtered list, 0 SQL
    for r in note.pending_reminders:
        print(r.remind_at)
```

> **Why `to_attr`?** Without it, Django stores prefetch results in the internal cache
> of `note.reminders.all()`. But any call to `note.reminders.filter(...)` creates a
> **new** QuerySet that bypasses the cache — causing extra SQL again!
>
> `to_attr='pending_reminders'` stores results as a plain Python **`list`**, so
> `note.pending_reminders` is always a zero-SQL memory read.

---

### Exists() Subquery — Boolean Optimization

For **boolean checks** without fetching any rows, `Exists()` is the most efficient
possible pattern:

```python
from django.db.models import Exists, OuterRef

# ❌ Slow: fetches full objects just to check truthiness
if Note.objects.filter(user=user, is_archived=True):
    ...

# ✅ Fast: SELECT (1) AS a FROM note WHERE ... LIMIT 1
if Note.objects.filter(user=user, is_archived=True).exists():
    ...

# ✅✅ Optimal for annotation — avoids a JOIN entirely:
notes = Note.objects.annotate(
    has_pending_reminder=Exists(
        Reminder.objects.filter(
            note=OuterRef('pk'),    # OuterRef links subquery to outer note.id
            is_sent=False
        )
    )
)
# SQL: SELECT note.*,
#      EXISTS(
#          SELECT 1 FROM reminder
#          WHERE reminder.note_id = note.id AND reminder.is_sent = false
#      ) AS has_pending_reminder
#      FROM note
# → 1 query, no JOIN, no COUNT(*), no Python loop

for note in notes:
    print(note.has_pending_reminder)   # True / False — from memory
```

> **`select_related` vs `Exists()`:** Use `select_related` when you need the *data*
> from the related object. Use `Exists()` when you only need a **boolean flag** — it
> avoids transferring any rows at all.

---

### Decision Diagram

```mermaid
graph TD
    A[Need related data in a loop?] --> B{Relationship type}
    B -->|"ForeignKey / OneToOne"| C[select_related]
    C --> D["1 SQL with JOIN — data in memory"]
    B -->|"ManyToMany / Reverse FK"| E{Need filtering or ordering?}
    E -->|No| F[prefetch_related]
    F --> G["2 SQL queries + Python join"]
    E -->|Yes| H["Prefetch(queryset=..., to_attr=...)"]
    H --> G
    B -->|"Boolean check only"| I[".exists() or Exists() subquery"]
    I --> J["SELECT 1 LIMIT 1 — no data transferred"]
```

---

### N+1 Detection Techniques

```python
# ── Development: manual query counting ──────────────────────────────────────
from django.db import connection, reset_queries
from django.conf import settings
settings.DEBUG = True

reset_queries()
# ... run your view code here ...
print(f"Total queries: {len(connection.queries)}")
for q in connection.queries:
    print(f"  [{q['time']}s] {q['sql'][:120]}")

# ── Development: queryset.explain() ─────────────────────────────────────────
# Ask PostgreSQL for the actual execution plan
print(Note.objects.select_related('notebook').explain(analyze=True))
# Output: Seq Scan vs Index Scan vs Hash Join

# ── Production: Django Debug Toolbar ────────────────────────────────────────
# pip install django-debug-toolbar
# → Shows per-request query count, duplicates, and slow queries in browser panel
```


In [ ]:
# ──────────────────────────────────────────────────
# N+1 Problem — підготовка та демонстрація
# ──────────────────────────────────────────────────

from django.db import connection, reset_queries
from django.conf import settings

# settings.DEBUG = True — вмикає логування SQL запитів у connection.queries.
# Без DEBUG=True connection.queries завжди буде порожнім списком.
settings.DEBUG = True

def count_queries(func):
    """Рахує та виводить всі SQL запити, що виконалися під час func()."""
    reset_queries()         # очищаємо лог попередніх запитів
    func()                  # виконуємо функцію — вона робить ORM-запити
    queries = connection.queries  # список dict: [{'sql': ..., 'time': ...}, ...]
    print(f'\n  📊 Всього SQL запитів: {len(queries)}')
    for i, q in enumerate(queries, 1):
        short = q['sql'][:100] + '...' if len(q['sql']) > 100 else q['sql']
        print(f'  [{i}] {q["time"]}s: {short}')

# ── N+1 PROBLEM: 1 SQL + N SQLs у циклі ──────────────────────────────────────
# Проблема: для кожної нотатки Django робить ОКРЕМИЙ запит до notebook.
# 10 нотаток = 11 SQL запитів (1 для notes + 10 для кожного notebook).
# На 1000 нотаток = 1001 SQL. Катастрофа!
print('=== ❌ N+1 PROBLEM (без оптимізації) ===')
def bad_query():
    # 1 SQL: SELECT * FROM hello_app_note WHERE user_id = ...
    notes = Note.objects.filter(user=lab_user)
    for note in notes:
        # +1 SQL КОЖНА нотатка!
        # Django завантажує note.notebook лише коли ти до нього звертаєшся.
        # Django: SELECT * FROM hello_app_notebook WHERE id = note.notebook_id
        _ = note.notebook.title if note.notebook else 'None'

count_queries(bad_query)

# ── select_related() РІШЕННЯ: всього 1 SQL JOIN ──────────────────────────────
# select_related('notebook') каже Django: зроби SQL JOIN і завантаж notebook разом.
# SQL: SELECT note.*, notebook.* FROM hello_app_note
#      JOIN hello_app_notebook ON note.notebook_id = notebook.id
#      WHERE note.user_id = ...
# Результат: note.notebook вже завантажений, без додаткових SQL.
print('\n=== ✅ select_related() — JOIN (1 SQL) ===')
def good_query_fk():
    # 1 SQL з JOIN — всі notebook завантажено відразу
    notes = Note.objects.filter(user=lab_user).select_related('notebook')
    for note in notes:
        _ = note.notebook.title if note.notebook else 'None'  # 0 SQL! Вже завантажено

count_queries(good_query_fk)
print(f'SQL: {sql(Note.objects.filter(user=lab_user).select_related("notebook"))}')


In [ ]:
# ──────────────────────────────────────────────────
# M2M N+1 — і його рішення prefetch_related()
# ──────────────────────────────────────────────────

# M2M (ManyToMany) — Django НЕ може зробити JOIN для M2M в select_related.
# Причина: join M2M дає дублікати рядків (нотатка × тег = N рядків).
# Рішення: prefetch_related() — 2 окремих SQL + Python join.

# ПРОБЛЕМА: 1 SQL для notes + N SQLs для tags кожної нотатки
print('=== ❌ N+1 для M2M (без оптимізації) ===')
def bad_m2m():
    # 1 SQL: SELECT * FROM hello_app_note WHERE user_id = ...
    notes = Note.objects.filter(user=lab_user)
    for note in notes:
        # +1 SQL КОЖНА нотатка!
        # Django: SELECT tag.* FROM hello_app_tag
        #         JOIN hello_app_note_tags ON ... WHERE note_id = note.id
        tags = list(note.tags.all())

count_queries(bad_m2m)

# РІШЕННЯ: prefetch_related() виконує 2 SQL, потім Python join у пам'яті.
# SQL 1: SELECT * FROM hello_app_note WHERE user_id = ...
# SQL 2: SELECT tag.*, note_tags.note_id FROM hello_app_tag
#         JOIN hello_app_note_tags ON ... WHERE note_tags.note_id IN (1,2,3,...)
# Python: Django сам розподіляє теги по нотатках у пам'яті.
# note.tags.all() всередині циклу — бере з Python-кешу (0 SQL)!
print('\n=== ✅ prefetch_related() — 2 SQL (Python join) ===')
def good_m2m():
    # 2 SQL: один для notes, один для всіх tags цих notes
    notes = Note.objects.filter(user=lab_user).prefetch_related('tags')
    for note in notes:
        tags = list(note.tags.all())  # 0 SQL! З prefetch кешу

count_queries(good_m2m)

# КОМБІНАЦІЯ: select_related для FK + prefetch_related для M2M
# SQL 1: SELECT note.*, notebook.* FROM note JOIN notebook ON ...
# SQL 2: SELECT tag.* ... WHERE note_id IN (...)
# SQL 3: SELECT user.* ... (якщо 'user' не в кеші)
print('\n=== ✅ Комбінований: select_related + prefetch_related ===')
def full_optimized():
    notes = (Note.objects
             .filter(user=lab_user)
             # select_related — JOIN для FK (notebook, user) → 1 SQL
             .select_related('notebook', 'user')
             # prefetch_related — окремий SELECT для M2M (tags) → 1 SQL
             .prefetch_related('tags'))
    for note in notes:
        _ = note.notebook.title if note.notebook else 'None'
        _ = list(note.tags.all())

count_queries(full_optimized)


In [ ]:
# ──────────────────────────────────────────────────
# Prefetch() — розширений контроль prefetch
# ──────────────────────────────────────────────────

from django.db.models import Prefetch

# Prefetch() — клас що дозволяє:
# 1) Передати власний QuerySet для prefetch (з фільтрами, only, annotate тощо)
# 2) Зберегти результат у кастомний атрибут через to_attr=
#
# Без Prefetch(): prefetch_related('tags') завантажить ВСІ теги для кожної нотатки.
# З Prefetch(): можна завантажити тільки теги поточного user + тільки потрібні поля.

print('=== Prefetch() об'єкт — фільтрований prefetch ===')

# Prefetch(
#   'tags',                                          ← назва M2M поля
#   queryset=Tag.objects.filter(...).only('name','color'),  ← кастомний QuerySet
#   to_attr='cached_tags'                            ← зберегти як звичайний список
# )
# SQL для prefetch: SELECT id, name, color FROM hello_app_tag
#                   JOIN hello_app_note_tags ON ...
#                   WHERE note_tags.note_id IN (...)
#                     AND hello_app_tag.user_id = lab_user.id
urgent_tags = Prefetch(
    'tags',
    queryset=Tag.objects.filter(user=lab_user).only('name', 'color'),
    to_attr='cached_tags'  # замість .tags.all() → note.cached_tags (звичайний list)
)

# prefetch_related(urgent_tags) — використовуємо Prefetch-об'єкт замість рядка.
notes = (Note.objects
         .filter(user=lab_user)
         .prefetch_related(urgent_tags))

for note in notes:
    # note.cached_tags — звичайний Python list (НЕ QuerySet, НЕ RelatedManager).
    # 0 SQL! Всі теги вже завантажені prefetch'ом і збережені у to_attr=.
    tag_names = [t.name for t in note.cached_tags]
    print(f'  {note.title}: {tag_names}')


---
> 📚 **Документація:** [DJANGO_ORM_DEEP.md](../../DJANGO_ORM_DEEP.md)
### 8.1 🔎 Exists() Subquery — Live Demo

In [ ]:
# ──────────────────────────────────────────────────
# Exists() — boolean subquery (підзапит EXISTS)
# ──────────────────────────────────────────────────

from django.db.models import Exists, OuterRef

# Exists() — Django-підзапит EXISTS у SQL.
# Використовується в annotate() щоб додати boolean поле до кожного запису.
#
# OuterRef('pk') — посилання на поле ЗОВНІШНЬОГО запиту (Note.pk).
# Це як SQL: WHERE note_id = outer_note.id
#
# Результуючий SQL (1 запит, підзапит EXISTS):
# SELECT note.*,
#   EXISTS(
#     SELECT 1 FROM hello_app_reminder r
#     WHERE r.note_id = note.id AND r.is_sent = FALSE
#   ) AS has_pending
# FROM hello_app_note
# WHERE user_id = lab_user.id
#
# Переваги Exists() над prefetch_related + Python:
# 1) 1 SQL запит замість 2
# 2) Не завантажує Reminder об'єкти в пам'ять
# 3) Тільки boolean (True/False) — мінімальний трафік

print('=== Exists() subquery — boolean annotation (1 SQL) ===')
reset_queries()

notes_with_flag = Note.objects.filter(user=lab_user).annotate(
    # has_pending — нове boolean поле для кожної нотатки.
    # Exists() → True якщо є хоч один Reminder з is_sent=False для цієї нотатки.
    has_pending=Exists(
        Reminder.objects.filter(
            note=OuterRef('pk'),  # note_id = зовнішній note.id
            is_sent=False
        )
    )
)

for note in notes_with_flag:
    flag = '⏰' if note.has_pending else '  '
    print(f'  {flag} [{note.priority}] {note.title[:40]}')

print(f'\n  📊 Queries used: {len(connection.queries)} (expected: 1)')

# SQL inspection — переглядаємо що Django згенерував
print('\n=== SQL generated by Exists() subquery ===')
print(notes_with_flag.query)

# .exists() (метод без Exists()) — перевіряє наявність хоч одного запису.
# SQL: SELECT 1 FROM hello_app_note WHERE ... LIMIT 1
# Найшвидший спосіб перевірити "чи є хоч щось?" — без завантаження даних.
print('\n=== .exists() — single check, fastest ===')
reset_queries()

has_any = Note.objects.filter(user=lab_user, priority=4).exists()
print(f'  Has critical notes: {has_any}')
print(f'  Queries: {len(connection.queries)}  ← SELECT 1 LIMIT 1')


---
> 📚 **Документація:** [DJANGO_ORM_DEEP.md](../../DJANGO_ORM_DEEP.md)
## 9. 🎛️ only() і defer() — Завантажуємо тільки потрібні колонки

```sql
-- only('title', 'priority'):
SELECT id, title, priority FROM note WHERE ...

-- defer('content'):
SELECT id, title, priority, notebook_id, ... FROM note WHERE ...  (без content)
```

> Корисно коли є великі `TextField` (content, description) які не потрібні в списку.

In [ ]:
# ──────────────────────────────────────────────────
# only() та defer() — завантажуємо тільки потрібні колонки
# values() та values_list() — словники/кортежі замість об'єктів
# ──────────────────────────────────────────────────

# only('title', 'priority', 'is_pinned') — Django включає ТІЛЬКИ ці колонки в SELECT.
# SQL: SELECT id, title, priority, is_pinned FROM hello_app_note WHERE user_id = ...
# ВАЖЛИВО: id завжди включається автоматично!
# Якщо звернутись до note.content — Django зробить ДОДАТКОВИЙ SQL запит.
# (деferred field) — це ловушка для початківців!
print('=== only() — завантажити тільки вказані поля ===')
qs = Note.objects.filter(user=lab_user).only('title', 'priority', 'is_pinned')
print(f'SQL: {sql(qs)}')
note = qs.first()
print(f'title: {note.title}')        # ✅ завантажено
print(f'priority: {note.priority}')  # ✅ завантажено
# print(f'content: {note.content}')  # ⚠️ запустить ДОДАТКОВИЙ SQL запит!

# defer('content') — протилежність only().
# Завантажуємо ВСЕ крім вказаних великих полів.
# SQL: SELECT id, title, priority, is_pinned, ... (всі КРІМ content)
# Корисно коли є великий TextField що не потрібен у списку.
print('\n=== defer() — не завантажувати великі поля ===')
qs = Note.objects.filter(user=lab_user).defer('content')  # content — TextField, великий
print(f'SQL: {sql(qs)}')
print('(content не в SELECT, завантажиться окремим запитом лише якщо звернутись)')

# values('title', 'priority', 'notebook__title') — повертає QuerySet dict-ів.
# НЕ Model instances! Кожен рядок — Python dict {'title': ..., 'priority': ...}.
# notebook__title — traversal через FK, Django робить JOIN автоматично.
# SQL: SELECT title, priority, notebook.title AS notebook__title
#      FROM hello_app_note JOIN hello_app_notebook ON ...
#      WHERE user_id = ...
print('\n=== values() — словники замість об'єктів ===')
qs = Note.objects.filter(user=lab_user).values('title', 'priority', 'notebook__title')
print(f'SQL: {sql(qs)}')
for row in qs:
    print(f'  {row}')  # row — звичайний dict

# values_list() — як values(), але повертає кортежі замість dict-ів.
# Ефективніше для великих наборів даних — менше пам'яті.
print('\n=== values_list() — кортежі ===')
qs = Note.objects.filter(user=lab_user).values_list('title', 'priority')
print(f'Результат: {list(qs)}')

# flat=True — тільки для ONE поля. Повертає плоский список рядків/чисел.
# Без flat=True: [('назва',), ('назва 2',), ...]
# З flat=True:  ['назва', 'назва 2', ...]
titles = Note.objects.filter(user=lab_user).values_list('title', flat=True)
print(f'values_list(flat=True): {list(titles)}')


---
> 📚 **Документація:** [DJANGO_ORM_DEEP.md](../../DJANGO_ORM_DEEP.md)
## 10. 🏗️ Custom Managers & QuerySets

Замість повторення однакових `filter()` у views — виносимо логіку в Manager/QuerySet.

```python
# Без custom manager (погано — дублювання!):
notes = Note.objects.filter(user=user, is_archived=False, priority__gte=3)
# В іншому view:
notes = Note.objects.filter(user=user, is_archived=False, priority__gte=3)

# З custom manager (добре!):
notes = Note.urgent.for_user(user)
```

In [ ]:
# ──────────────────────────────────────────────────
# Custom Managers & QuerySets
# Reusable фільтри як методи QuerySet
# ──────────────────────────────────────────────────

# Демонстрація через динамічне створення Manager прямо в notebook.
# У production: NoteQuerySet живе в models.py і підключається через objects = NoteQuerySet.as_manager()
from django.db.models import Manager, QuerySet as DjangoQuerySet

# QuerySet subclass — кожен метод повертає self або self.filter(...)
# Це дозволяє chaining: .for_user(u).active().urgent()
# Кожен метод додає умови через AND, НЕ виконує SQL.
class NoteQuerySet(DjangoQuerySet):
    """Reusable QuerySet методи для Note"""

    def active(self):
        """Не архівовані нотатки. SQL: ... AND is_archived = FALSE"""
        return self.filter(is_archived=False)

    def urgent(self):
        """Термінові нотатки (priority=4). SQL: ... AND priority = 4"""
        return self.filter(priority=4)

    def high_or_urgent(self):
        """Пріоритет 3 або 4. SQL: ... AND priority >= 3"""
        return self.filter(priority__gte=3)

    def for_user(self, user):
        """Ізоляція по user. SQL: ... AND user_id = user.id"""
        return self.filter(user=user)

    def with_full_data(self):
        """Оптимізований QuerySet з усіма зв'язками.
        Додає SELECT + JOIN + prefetch — але все ще LAZY."""
        return self.select_related('notebook', 'user').prefetch_related('tags')


# as_manager() — конвертує QuerySet у Manager.
# Manager — це точка входу (Note.objects). 
# QuerySet — це сам запит що будується.
NoteManager = NoteQuerySet.as_manager()

# Демо — будуємо chain методів:
# NoteQuerySet(model=Note) — починаємо із "пустого" QuerySet для Note.
# .for_user() → .active() → .high_or_urgent() → .with_full_data()
# Кожен метод повертає новий QuerySet з доданими умовами.
# SQL виконується ТІЛЬКИ коли ітеруємо результат (в кінці).
base_qs = NoteQuerySet(model=Note)

qs = base_qs.for_user(lab_user).active().high_or_urgent().with_full_data()
print('=== Custom Manager chain ===')
print(f'SQL: {sql(qs)}')
print(f'Результат: {[(n.title, n.priority) for n in qs]}')


---
> 📚 **Документація:** [INDEXING_DEEP.md](../../INDEXING_DEEP.md) · [POSTGRESQL_ADVANCED.md](../../POSTGRESQL_ADVANCED.md)
## 11. 🔬 SQL Inspection — Дебаг запитів

Для розуміння що Django насправді виконує:

In [ ]:
# ──────────────────────────────────────────────────
# SQL Inspection — дебаг QuerySet
# ──────────────────────────────────────────────────

# qs.query — властивість QuerySet що показує SQL без виконання запиту.
# Повертає: django.db.models.sql.query.Query об'єкт.
# str(qs.query) — перетворює на рядок SQL (наш helper sql() робить те саме).
# ВАЖЛИВО: .query НЕ виконує SQL — це просто інтроспекція.
# Але деякі частини (prefetch_related) НЕ відображаються в .query (вони окремий SQL).
print('=== .query — переглянути SQL без виконання ===')

qs = (
    Note.objects
    .filter(user=lab_user, priority__gte=2)
    # select_related — JOIN у головний SQL (видно в .query)
    .select_related('notebook')
    # prefetch_related — окремий SQL (НЕ видно в .query!)
    .prefetch_related('tags')
    .order_by('-priority', 'title')  # ORDER BY priority DESC, title ASC
    .distinct()                        # DISTINCT
)
# str(qs.query) показує ТІЛЬКИ головний SQL без prefetch.
# Відформатовано для читання Django.
print(qs.query)

# explain() — показує план виконання запиту (EXPLAIN від БД).
# PostgreSQL: виводить cost=, rows=, Index Scan vs Seq Scan
# SQLite 3.31+: виводить SCAN TABLE vs SEARCH TABLE
# ДУЖЕ корисно для оптимізації: бачиш чи використовується index!
print('\n=== explain() — план виконання ===')
try:
    # explain() виконує РЕАЛЬНИЙ запит до БД (не просто SQL рядок).
    # Django під капотом: EXPLAIN SELECT * FROM hello_app_note WHERE ...
    plan = Note.objects.filter(user=lab_user).explain()
    print(plan)
except Exception as e:
    print(f'explain() вимагає PostgreSQL або SQLite 3.31+: {e}')


In [ ]:
# ──────────────────────────────────────────────────
# connection.queries — лог всіх виконаних SQL запитів
# ──────────────────────────────────────────────────

# connection.queries — список dict-ів з усіма SQL запитами поточної сесії.
# Кожен dict: {'sql': 'SELECT ...', 'time': '0.001'}
# Заповнюється тільки якщо settings.DEBUG = True (вже встановлено вище).
# reset_queries() — очищає список (скидаємо лічильник).
print('=== connection.queries — всі SQL запити сесії ===')

reset_queries()  # починаємо з чистого списку

# Виконуємо кілька ORM-операцій:

# list() — примушує виконати SQL. select_related додає JOIN.
# SQL 1: SELECT note.*, notebook.* FROM hello_app_note JOIN hello_app_notebook ON ...
_ = list(Note.objects.filter(user=lab_user).select_related('notebook'))

# .first() — виконує SQL. annotate() додає GROUP BY.
# SQL 2: SELECT notebook.*, COUNT(note.id) AS cnt FROM ... GROUP BY notebook.id LIMIT 1
_ = Notebook.objects.filter(user=lab_user).annotate(cnt=Count('notes')).first()

# .count() — виконує SQL. SELECT COUNT(*).
# SQL 3: SELECT COUNT(*) FROM hello_app_tag WHERE user_id = ...
_ = Tag.objects.filter(user=lab_user).count()

# connection.queries тепер містить 3 записи.
# Кожен — dict: {'sql': ..., 'time': '0.001'}
# 'time' — час виконання в секундах (рядок!).
# Корисно для профілювання: які запити найповільніші?
print(f'Виконано {len(connection.queries)} запитів:')
for i, q in enumerate(connection.queries, 1):
    print(f'\n[{i}] Час: {q["time"]}s')
    print(f'     SQL: {q["sql"][:200]}')


---
> 📚 **Документація:** [DJANGO_ORM_DEEP.md](../../DJANGO_ORM_DEEP.md)
## 12. 💥 Edge Cases — DoesNotExist, MultipleObjectsReturned, distinct()

Критичні помилки які потрібно знати:

In [ ]:
# ──────────────────────────────────────────────────
# Edge Cases — типові помилки початківців
# DoesNotExist, MultipleObjectsReturned, distinct()
# ──────────────────────────────────────────────────

# get() — очікує РІВНО ОДИН запис.
# Якщо не знайдено → піднімає Note.DoesNotExist (підклас ObjectDoesNotExist).
# SQL: SELECT * FROM hello_app_note WHERE user_id=... AND title='...' LIMIT 21
# (Django перевіряє якщо знайдено > 1 → MultipleObjectsReturned)
print('=== DoesNotExist ===')
try:
    # Навмисно шукаємо нотатку якої немає → буде DoesNotExist.
    note = Note.objects.get(user=lab_user, title='Неіснуюча нотатка 404')
except Note.DoesNotExist:
    print('❌ DoesNotExist: get() не знайшов жодного запису')
    print('   Рішення: використати .first() або .filter().exists()')

# .first() — безпечна альтернатива get() коли запис може не існувати.
# SQL: SELECT * FROM hello_app_note WHERE ... LIMIT 1
# Повертає: Model instance АБО None (не піднімає Exception).
note = Note.objects.filter(user=lab_user, title='Неіснуюча нотатка 404').first()
print(f'   .first() = {note}  (None замість Exception)')

# get() — якщо знайдено > 1 запис → MultipleObjectsReturned.
# Типова помилка: filter без достатньо унікальних умов.
print('\n=== MultipleObjectsReturned ===')
try:
    # get(user=lab_user) — у нас 10 нотаток для цього user.
    # Django знаходить > 1 запис → MultipleObjectsReturned.
    note = Note.objects.get(user=lab_user)
except Note.MultipleObjectsReturned:
    count = Note.objects.filter(user=lab_user).count()
    print(f'❌ MultipleObjectsReturned: get() знайшов {count} записів, очікував 1')
    print('   Рішення: filter().first() або додати унікальний фільтр')

# distinct() — видаляє дублікати рядків у результаті.
# M2M JOIN може дати дублікати: нотатка з 2 тегами з'явиться 2 рази після JOIN.
# .distinct() додає SQL: SELECT DISTINCT ...
print('\n=== distinct() для M2M joins ===')
# Фільтруємо нотатки що мають хоч 1 тег (tags__isnull=False → JOIN).
# Після JOIN нотатка з 3 тегами з'явиться 3 рази!
qs_dup = Note.objects.filter(user=lab_user, tags__isnull=False)
qs_uniq = qs_dup.distinct()
print(f'Без distinct(): {qs_dup.count()} рядків')
print(f'З distinct():   {qs_uniq.count()} унікальних нотаток')


---
> 📚 **Документація:** [TRANSACTIONS_CONCURRENCY.md](../../TRANSACTIONS_CONCURRENCY.md) · [DJANGO_ORM_DEEP.md](../../DJANGO_ORM_DEEP.md)
## 13. 🔒 Транзакції — atomic()

**Транзакція** = група операцій яка або виконується ПОВНІСТЮ, або відкочується (ROLLBACK).

```
transaction.atomic():
  ┌─────────────────────────────────┐
  │  BEGIN TRANSACTION              │
  │  INSERT INTO note ...           │
  │  INSERT INTO note_tags ...      │
  │  UPDATE notebook SET count...   │
  │  COMMIT  ← якщо все ОК         │
  │  ROLLBACK ← якщо Exception     │
  └─────────────────────────────────┘
```

In [ ]:
# ──────────────────────────────────────────────────
# transaction.atomic() — групові операції
# "Або всі, або нічого"
# ──────────────────────────────────────────────────

from django.db import transaction, IntegrityError

# Прибираємо залишки попередніх запусків цієї клітинки.
# (при повторному запуску нотебуку дані вже можуть бути в БД)
Note.objects.filter(
    user=lab_user,
    title__in=[
        "Транзакційна нотатка",
        "Невалідна нотатка",
        "Зовнішня нотатка",
        "Внутрішня нотатка",
    ],
).delete()

# transaction.atomic() — відкриває SQL-транзакцію.
# Django: BEGIN TRANSACTION
# Все що відбувається всередині with-блоку — в одній транзакції.
# При успіху → COMMIT (всі зміни зберігаються).
# При Exception → ROLLBACK (жодна зміна не зберігається).
print("=== transaction.atomic() — групова операція ===")

with transaction.atomic():
    # Note.objects.create() — INSERT INTO hello_app_note VALUES (...)
    # Виконується в межах транзакції. Якщо помилка нижче → цей рядок теж відкатиться!
    new_note = Note.objects.create(
        user=lab_user,
        title="Транзакційна нотатка",
        content="Створено в транзакції",
        priority=2,
    )

    # get_or_create() — атомарний GET або CREATE.
    # SQL: спочатку SELECT, якщо не знайдено → INSERT.
    # Повертає: (tag_instance, created_bool)
    # created = True якщо тег щойно створено, False якщо вже існував.
    new_tag, _ = Tag.objects.get_or_create(
        user=lab_user,
        name="atomic",
        defaults={"color": "#e91e63"},  # тільки при CREATE
    )

    # new_note.tags.add(new_tag) — INSERT INTO hello_app_note_tags (note_id, tag_id)
    new_note.tags.add(new_tag)

    print(f"✅ Нотатку #{new_note.pk} створено з тегом #{new_tag.name}")


# ROLLBACK демонстрація — IntegrityError всередині atomic() = відкат всіх змін.
print("\n=== ROLLBACK при помилці бази даних ===")

count_before = Note.objects.filter(
    user=lab_user,
    title__startswith="Невалідна",
).count()

try:
    with transaction.atomic():
        # priority=99 — порушує CheckConstraint з models.py (priority між 1 і 4).
        # Django генерує IntegrityError → ROLLBACK всієї транзакції.
        Note.objects.create(
            user=lab_user,
            title="Невалідна нотатка",
            content="Цей запис не має зберегтися",
            priority=99,  # ← невалідне значення!
        )

except IntegrityError as e:
    print(f"❌ IntegrityError: {str(e)[:80]}")
    count_after = Note.objects.filter(
        user=lab_user,
        title__startswith="Невалідна",
    ).count()
    print(f"   Записів до: {count_before}, після: {count_after} (ROLLBACK спрацював)")


---
> 📚 **Документація:** [POSTGRESQL_ADVANCED.md](../../POSTGRESQL_ADVANCED.md) · [RELATIONAL_DB_FOUNDATIONS.md](../../RELATIONAL_DB_FOUNDATIONS.md)
## 14. 🧪 Raw SQL — Коли ORM недостатньо

Інколи потрібен складний SQL якого ORM не підтримує (рекурсивні CTE, складні window functions тощо).  

⚠️ **Завжди використовуй параметризовані запити!** SQL Injection = критична вразливість.

In [ ]:
# ──────────────────────────────────────────────────
# Raw SQL — коли ORM недостатньо
# raw() та cursor() — два способи
# ──────────────────────────────────────────────────

from django.db import connection

# raw() — власний SQL але з маппінгом результатів на Model instances.
# Повертає RawQuerySet — Django читає колонки і перетворює на Note об'єкти.
# БЕЗПЕКА: параметри передаються ОКРЕМО другим аргументом!
# НЕ МОЖНА: raw(f"WHERE user_id = {lab_user.pk}") ← SQL Injection!
# МОЖНА:    raw("WHERE user_id = %s", [lab_user.pk]) ← безпечно, параметризовано
print('=== raw() — ORM + власний SQL ===')
raw_qs = Note.objects.raw(
    'SELECT * FROM hello_app_note WHERE user_id = %s AND priority >= %s ORDER BY priority DESC',
    [lab_user.pk, 3]  # %s → значення, Django екранує їх
)
# raw_qs — RawQuerySet. Ітерація виконує SQL і маппить рядки на Note.
for note in raw_qs:
    # note — повноцінний Note instance. note.title, note.priority — доступні.
    print(f'  [{note.priority}] {note.title}')

# cursor() — прямий доступ до БД без маппінгу на моделі.
# Використовується для: складних JOIN, WINDOW-функцій, GROUP BY, агрегації.
# Повертає: список кортежів (або dict після fetchall + description).
# cursor.description — метаінформація про колонки результату.
print('\n=== cursor() — прямий SQL (без маппінгу на моделі) ===')
with connection.cursor() as cursor:
    # cursor.execute(sql, params) — параметризований запит.
    # %s — placeholder. Django замінює на реальні значення (екрановано!).
    cursor.execute(
        '''
        SELECT
            n.title,
            n.priority,
            nb.title as notebook_name,
            COUNT(nt.tag_id) as tag_count
        FROM hello_app_note n
        LEFT JOIN hello_app_notebook nb ON n.notebook_id = nb.id
        LEFT JOIN hello_app_note_tags nt ON n.id = nt.note_id
        WHERE n.user_id = %s
        GROUP BY n.id, n.title, n.priority, nb.title
        ORDER BY n.priority DESC
        ''',
        [lab_user.pk]
    )
    # fetchall() — повертає всі рядки як список кортежів.
    rows = cursor.fetchall()
    # cursor.description — список RealDictRow з назвами колонок.
    cols = [desc[0] for desc in cursor.description]

# rows — список кортежів. zip(cols, row) → dict для зручного читання.
print(f'Колонки: {cols}')
for row in rows:
    d = dict(zip(cols, row))
    print(f'  {d["title"]:30} p={d["priority"]}  nb={d["notebook_name"]}  tags={d["tag_count"]}')


---
> 📚 **Документація:** [DJANGO_ORM_DEEP.md](../../DJANGO_ORM_DEEP.md)
## 15. 📋 Bulk операції — Масові INSERT/UPDATE/DELETE

| Метод | SQL | Коли |
|-------|-----|------|
| `bulk_create()` | 1 `INSERT` для N записів | Масовий імпорт |
| `bulk_update()` | 1 `UPDATE` для N записів | Масове оновлення |
| `update()` | `UPDATE WHERE` | Оновити всі що відповідають фільтру |
| `delete()` | `DELETE WHERE` | Видалити всі що відповідають фільтру |

In [ ]:
# ──────────────────────────────────────────────────
# Bulk операції — масові INSERT/UPDATE/DELETE
# Замість N окремих SQL — 1 ефективний запит
# ──────────────────────────────────────────────────

# bulk_create() — масовий INSERT одним запитом.
# Без bulk_create: 5 нотаток = 5 окремих INSERT (повільно!).
# З bulk_create: 1 INSERT з 5 VALUES у одному SQL.
# SQL: INSERT INTO hello_app_note (user_id, title, priority)
#      VALUES (1, 'Bulk нотатка 1', 2), (1, 'Bulk нотатка 2', 3), ...
print('=== bulk_create() — масовий INSERT ===')
reset_queries()  # скидаємо лічильник для чистого підрахунку

# Список unsaved Model instances (без pk — pk присвоїть БД після INSERT).
# Note(...) — конструктор моделі, НЕ зберігає в БД.
bulk_notes = [
    Note(user=lab_user, title=f'Bulk нотатка {i}', priority=(i % 4) + 1)
    for i in range(1, 6)  # 5 нотаток
]
# bulk_create() — один SQL INSERT для всіх 5 нотаток.
# Повертає список створених об'єктів з pk (якщо БД підтримує RETURNING).
created = Note.objects.bulk_create(bulk_notes)
print(f'Створено {len(created)} нотаток за 1 SQL запит')
print(f'SQL: {connection.queries[-1]["sql"][:150]}...')

# update() — масовий UPDATE одним SQL.
# Без update(): треба завантажити кожен об'єкт → змінити → .save() (N SQLs).
# З update(): 1 SQL UPDATE з WHERE умовою.
# SQL: UPDATE hello_app_note SET priority = 1 WHERE user_id=... AND title LIKE 'Bulk%'
print('\n=== update() — масовий UPDATE ===')
updated = Note.objects.filter(user=lab_user, title__startswith='Bulk').update(priority=1)
print(f'Оновлено {updated} нотаток за 1 SQL')
print(f'SQL: {connection.queries[-1]["sql"]}')

# delete() — масовий DELETE одним SQL.
# Повертає: (total_deleted, {ModelName: count}) — для інформації.
# Якщо є CASCADE — Django видаляє пов'язані об'єкти теж (автоматично).
# SQL: DELETE FROM hello_app_note WHERE user_id=... AND title LIKE 'Bulk%'
print('\n=== delete() — масовий DELETE ===')
deleted_count, deleted_types = Note.objects.filter(user=lab_user, title__startswith='Bulk').delete()
print(f'Видалено: {deleted_count} ({deleted_types})')
print(f'SQL: {connection.queries[-1]["sql"]}')


---
> 📚 **Документація:** [ORM_MERMAID.md](../../ORM_MERMAID.md) · [DJANGO_ORM_DEEP.md](../../DJANGO_ORM_DEEP.md)
## 16. 🗂️ QuerySet Lifecycle — Повна картина

```mermaid
graph TD
    A["Note.objects.filter(priority=4)"] -->|Lazy| B[QuerySet AST]
    B --> C[".exclude(), .order_by(), .select_related()"]
    C -->|Ще lazy| D["QuerySet з _result_cache=None"]
    D --> E{Trigger?}
    E -->|for note in qs| F[SQL Compiler]
    E -->|list / bool / len| F
    E -->|qs-index qs-slice| F
    F --> G[SQL String]
    G --> H[(Database)]
    H --> I[Python Model Instances]
    I --> J["_result_cache заповнений"]
    J -->|Повторна ітерація| K[З кешу без SQL]
```

---

## 17. 🎯 Підсумок: Правила продуктивності

| # | Правило | Приклад |
|---|---------|--------|
| 1 | Перевіряй наявність через `exists()` | `if qs.exists():` не `if qs:` |
| 2 | Рахуй через `count()` | `qs.count()` не `len(qs)` |
| 3 | FK → `select_related()` | `Note.objects.select_related('notebook')` |
| 4 | M2M / reverse FK → `prefetch_related()` | `.prefetch_related('tags')` |
| 5 | Не завантажуй великі поля → `defer()` / `only()` | `.defer('content')` |
| 6 | Масові зміни → `update()` / `bulk_create()` | `.update(priority=1)` |
| 7 | M2M фільтр → `distinct()` | `.filter(tags__...).distinct()` |
| 8 | Атомарні зміни → `F()` | `.update(priority=F('priority')+1)` |
| 9 | Дивися SQL → `.query` / Debug Toolbar | `print(qs.query)` |
| 10 | Raw SQL з параметрами | `raw(sql, [params])` ніколи `f"{user_input}"` |

In [ ]:
# ──────────────────────────────────────────────────
# 🏆 Фінальний запит — всі оптимізації разом
# Підсумок лабораторії
# ──────────────────────────────────────────────────

print('=== 🏆 Фінальний запит: всі оптимізації разом ===')

reset_queries()

# Будуємо QuerySet з УСІМА техніками що вивчили:
result = (
    Note.objects
    # 1. filter() — ізоляція по user (WHERE user_id = ...)
    .filter(user=lab_user)
    # 2. Q() — складна умова з OR (WHERE priority>=2 OR is_pinned=TRUE)
    .filter(
        Q(priority__gte=2) |
        Q(is_pinned=True)
    )
    # 3. exclude() — NOT умова (AND NOT is_archived=TRUE)
    .exclude(is_archived=True)
    # 4. select_related() — JOIN для FK без N+1 (notebook завантажено одразу)
    .select_related('notebook')
    # 5. Prefetch() — кастомний prefetch для M2M (тільки name, color)
    #    to_attr='tag_list' → note.tag_list (звичайний list, 0 SQL у циклі)
    .prefetch_related(
        Prefetch('tags', queryset=Tag.objects.only('name', 'color'), to_attr='tag_list')
    )
    # 6. annotate() — обчислене поле tag_count для кожного запису
    #    SQL: COUNT(tags) з DISTINCT (бо може бути join dups)
    .annotate(tag_count=Count('tags', distinct=True))
    # 7. only() — мінімальний SELECT (тільки потрібні поля, без content і т.д.)
    .only('title', 'priority', 'is_pinned', 'notebook_id')
    # 8. order_by() — сортування: спочатку найвищий priority, потім за алфавітом
    .order_by('-priority', 'title')
    # 9. distinct() — без дублів (M2M JOIN може давати дублікати)
    .distinct()
)

# sql(result) — показує MAIN SQL (без prefetch частини).
# prefetch_related виконує ОКРЕМИЙ SQL після головного.
print(f'SQL: {sql(result)}')
print()

# Ітерація — ТУТ виконуються SQL запити:
# SQL 1: головний SELECT з JOIN та WHERE
# SQL 2: prefetch SELECT для tags
for note in result:
    # note.notebook — вже в пам'яті (select_related) → 0 SQL
    nb_name = note.notebook.title if note.notebook else '—'
    # note.tag_list — вже в пам'яті (Prefetch to_attr) → 0 SQL
    tags = ', '.join(f'#{t.name}' for t in note.tag_list)
    pin = '📌 ' if note.is_pinned else ''
    # note.tag_count — вже обчислено (annotate) → 0 SQL
    print(f'  [{note.priority}] {pin}{note.title} | 📁 {nb_name} | 🏷️  {tags or "немає"}')

print(f'\n📊 Виконано {len(connection.queries)} SQL запитів (замість ~{result.count() + result.count()} без оптимізацій)')
print('💡 Без оптимізацій: 1 (notes) + N (notebook) + N (tags) запитів')
print('💡 З оптимізаціями: 2 запити незалежно від кількості нотаток!')


---

## ✅ Що ти вивчив у цій лабораторії

У цій лабораторії ти пройшов повний цикл роботи з **Django ORM**: від базового `QuerySet` до оптимізації SQL-запитів, транзакцій і масових операцій.

### 🧠 1. QuerySet Mental Model

- `QuerySet` — це **опис запиту**, а не список результатів.
- Він працює як **lazy AST**: Django будує запит у памʼяті, але не виконує SQL одразу.
- SQL виконується тільки тоді, коли результат реально потрібен.

#### SQL виконується при:

```python
for note in qs
list(qs)
bool(qs)
len(qs)
qs[0]
qs.count()
qs.exists()
qs.aggregate()
````

---

### 🔍 2. Базова фільтрація

* `filter()` створює умови через `AND`.
* `exclude()` створює заперечення через `NOT`.
* `Q()` дозволяє будувати складні умови: `OR`, `AND`, `NOT`.

```python
Note.objects.filter(priority=4)

Note.objects.exclude(is_archived=True)

Note.objects.filter(
    Q(priority=4) | Q(is_pinned=True)
)
```

---

### 🔗 3. Relationship Traversal

Django використовує подвійне підкреслення `__` для переходу між моделями.

```python
Note.objects.filter(notebook__title="Django-проєкт")
Note.objects.filter(tags__name="urgent")
```

Це означає:

```text
Python ORM query
        ↓
Django будує JOIN
        ↓
SQL звертається до повʼязаних таблиць
```

---

### ⚡ 4. Оптимізація запитів

Ти розібрав головну проблему продуктивності Django ORM — **N+1 queries**.

#### Правило:

| Тип звʼязку               | Інструмент           | Що робить                            |
| ------------------------- | -------------------- | ------------------------------------ |
| `ForeignKey` / `OneToOne` | `select_related()`   | Один SQL-запит через `JOIN`          |
| `ManyToMany` / Reverse FK | `prefetch_related()` | Два SQL-запити + обʼєднання в Python |

```python
Note.objects.select_related("notebook")

Note.objects.prefetch_related("tags")
```

---

### 📊 5. Aggregation & Annotation

Ти вивчив різницю між `aggregate()` і `annotate()`.

| Метод         | Логіка                       | Результат                     |
| ------------- | ---------------------------- | ----------------------------- |
| `aggregate()` | Підсумок для всього QuerySet | Один словник                  |
| `annotate()`  | Обчислення для кожного рядка | QuerySet з додатковими полями |

```python
Note.objects.aggregate(avg_priority=Avg("priority"))

Notebook.objects.annotate(note_count=Count("notes"))
```

---

### 🧮 6. F() Expressions

`F()` дозволяє виконувати операції **на рівні бази даних**, без завантаження значення в Python.

```python
Note.objects.filter(id=1).update(
    priority=F("priority") + 1
)
```

Це важливо, бо:

* операція атомарна;
* немає race condition;
* не потрібно робити окремий `SELECT`;
* база сама оновлює значення поля.

---

### 📦 7. Bulk Operations

Ти побачив, як виконувати масові операції ефективно.

```python
Note.objects.bulk_create([...])

Note.objects.filter(...).update(priority=1)

Note.objects.filter(...).delete()
```

#### Головна ідея:

```text
Не 100 окремих SQL-запитів,
а 1 масова операція.
```

---

### 🧹 8. distinct() після M2M

При фільтрації через `ManyToMany` можуть зʼявлятися дублікати.

```python
Note.objects.filter(
    tags__name__in=["django", "database"]
).distinct()
```

`distinct()` прибирає дублікати на рівні SQL.

---

### 🔒 9. Transactions

Ти вивчив `transaction.atomic()`:

```python
with transaction.atomic():
    note = Note.objects.create(...)
    note.tags.add(tag)
```

Транзакція означає:

```text
або всі операції виконались успішно,
або база відкотила все назад.
```

---

### 🧨 10. Raw SQL

Raw SQL можна використовувати, але тільки безпечно.

#### ❌ Погано:

```python
cursor.execute(f"SELECT * FROM note WHERE title = '{user_input}'")
```

#### ✅ Правильно:

```python
cursor.execute(
    "SELECT * FROM note WHERE title = %s",
    [user_input]
)
```

Головне правило:

> **Ніколи не вставляй user input у SQL через f-string.**

---

## 🧭 Головна архітектурна ідея

Django ORM — це не просто “зручний синтаксис для SQL”.

Це шар, який:

```text
Python objects
      ↓
QuerySet
      ↓
SQL compiler
      ↓
Database
      ↓
Model instances
```

Твоя задача як backend developer — не просто писати `.filter()`, а розуміти:

* коли SQL реально виконується;
* скільки SQL-запитів створюється;
* де виникає N+1;
* коли потрібен `JOIN`;
* коли потрібен `prefetch`;
* коли ORM достатньо;
* коли потрібен Raw SQL.

---

## 🚀 Наступні кроки

### 1. Django Debug Toolbar

Використай `Django Debug Toolbar`, щоб бачити SQL-запити прямо в браузері.

```python
INSTALLED_APPS = [
    ...
    "debug_toolbar",
]
```

Це допоможе бачити:

* кількість SQL-запитів;
* дублікати запитів;
* час виконання;
* N+1 problem;
* запити до шаблонів і views.

---

### 2. Silk Profiler

`django-silk` можна використати для глибшого profiling:

* час виконання view;
* SQL-запити;
* middleware;
* request/response аналіз;
* bottlenecks у backend.

---

### 3. PostgreSQL EXPLAIN ANALYZE

Коли перейдеш на PostgreSQL, наступний рівень — аналіз плану виконання:

```sql
EXPLAIN ANALYZE
SELECT ...
```

Це покаже:

* чи використовується індекс;
* чи база робить `Seq Scan`;
* які JOIN дорогі;
* де потрібно додати індекси.

---

### 4. django-query-count

Для CI/CD можна додати перевірку кількості SQL-запитів.

Ідея:

```text
якщо view раптом почав робити 100 SQL-запитів замість 5,
тест має впасти.
```

Це захищає проєкт від прихованих N+1 regression.

---

## 🏁 Підсумок

Після цієї лабораторії ти вже не просто “користуєшся ORM”.

Ти починаєш бачити Django ORM як систему:

```text
Model → QuerySet → SQL → Database → Optimization → Architecture
```

Це і є перехід від beginner-рівня до мислення backend engineer.


